# Notebook 5: Structure-Based Docking (GNINA + AutoDock Vina Consensus)

Structure-based docking layer for the GPCR benchmarking project — takes the
high-confidence, dataset-novel DrugBank candidates identified in notebook 4
and evaluates them structurally against each target's receptor.

**Scope:**
- 10 production receptor structures across the 5 targets (2 per target,
  except CCR5 which uses 2 allosteric-pocket structures instead of an
  active/inactive pair — see Module A). Every structure was live-verified
  against RCSB directly (species, chains, ligand identity, mutation counts,
  fusion-protein insertion sites) before being locked in, not assumed from
  memory or literature summaries.
- Consensus scoring: AutoDock Vina (empirical scoring function) + GNINA
  (CNN-based rescoring) — two genuinely different scoring philosophies, so
  agreement between them is a meaningful signal, not two similar tools
  agreeing with themselves. Both engines dock against the identical
  explicit box (same center, same size) so consensus reflects real scoring
  disagreement, not mismatched search volumes.
- Redocking validation is a hard gate, not a formality: up to 6 attempts per
  unit, stop at first RMSD <= 2.0 A, every attempt logged even after the
  loop exits early. A unit that never passes is flagged and skipped, not
  silently used.
- Per-unit isolation: each (target, state/pocket, engine) is checkpointed
  independently. One unit's failure never aborts any other unit.
- A lightweight, coded interaction-fingerprint (PLIF-style) plausibility
  filter runs after docking — not a full Discovery-Studio-style analysis,
  just auditable checks (key polar contacts, aromatic contacts, proximity
  to conserved anchor residues) to catch poses that scored well but don't
  make chemical sense.

**What this notebook does NOT do:**
- No new receptor-structure selection happens here — that decision is
  already made and disclosed in Module A's design matrix.
- No enzyme catalytic-site docking, no non-GPCR targets.
- No blind docking anywhere — every pocket has an experimentally-determined
  reference ligand position to center the search box on.


In [1]:
# MUST BE FIRST CELL!
import os
import sys
import multiprocessing
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
HPC_MODE = os.environ.get('HPC_MODE', '').lower() in ('1', 'true', 'yes') or not IN_COLAB

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/My Drive/gpcr_benchmark')
else:
    PROJECT_DIR = Path('./')

# Same mkdir placement as 06_docking.ipynb -- create PROJECT_DIR's subdirs
# right after mount, before anything else touches disk, so a fresh Drive
# (no prior notebook has run there yet) doesn't fail the assert below.
for _subdir in ['data/raw', 'data/processed', 'data/external/drugbank',
                 'ml/results', 'ml/models',
                 'docking/protein', 'docking/ligands', 'docking/logs',
                 'docking/docking_results', 'docking/figures']:
    (PROJECT_DIR / _subdir).mkdir(parents=True, exist_ok=True)

# Scheduler allocation exposed to the notebook, same precedence order used in
# notebooks 3/4: real PBS/SLURM env vars before falling back to cpu_count().
N_CORES = int(os.environ.get('NCPUS') or os.environ.get('PBS_NP') or
              os.environ.get('PBS_NCPUS') or os.environ.get('SLURM_CPUS_PER_TASK') or
              multiprocessing.cpu_count())

# Docking (Vina/GNINA) is not a joblib-parallel Python workload the way
# notebook 3's Optuna tuning was -- each engine call is its own subprocess
# and manages its own internal threading. Give each subprocess the full
# core count; do not fix these to 1 the way notebook 3 did for its
# many-small-fits-in-parallel pattern.
os.environ.setdefault('VINA_CPU', str(N_CORES))

print(f'IN_COLAB={IN_COLAB}  HPC_MODE={HPC_MODE}  N_CORES={N_CORES}')
print(f'PROJECT_DIR={PROJECT_DIR}')
assert PROJECT_DIR.exists(), f'PROJECT_DIR does not exist: {PROJECT_DIR}'

Mounted at /content/drive
IN_COLAB=True  HPC_MODE=False  N_CORES=2
PROJECT_DIR=/content/drive/My Drive/gpcr_benchmark


In [2]:
# =============================================================================
# INSTALL: OpenBabel (format conversion), Vina + Meeko (ligand/receptor prep),
# PDBFixer + OpenMM (receptor structure prep -- missing residues/atoms,
# forcefield-placed hydrogens, minimisation), GNINA (fetched as the latest
# GitHub release binary -- no pip package exists). Copied verbatim from
# 06_docking.ipynb's proven install cell, only extended at the bottom to
# also put vina/gnina on PATH since this notebook calls them as literal
# 'vina'/'gnina' subprocess commands rather than via GNINA_PATH.
# =============================================================================
!apt-get update -qq && apt-get install -y -q openbabel libboost-all-dev swig > /dev/null
# swig: required to build the 'vina' PyPI package's C++ Python bindings from
# source. Confirmed live (2026-08) that pip install vina can fail with a
# generic 'bdist_wheel did not run successfully' error on a fresh Colab
# image -- root cause is that vina's PyPI package has no pre-built wheel for
# every Python version, so a Colab image bump (out of our control, happens
# silently) can force a from-source build, and this project's install cell
# was missing swig, a documented AutoDock Vina build dependency. Was working
# before purely because whatever Python version Colab shipped at the time
# happened to have a pre-built wheel available, masking the missing swig.
# gemmi: meeko's chemtempgen module imports it directly but pip doesn't pull
# it in as a declared dependency -- installed explicitly to avoid a
# ModuleNotFoundError the first time meeko is actually imported.
# pdbfixer: pulls openmm>=8.2 in automatically as its own dependency -- no
# separate openmm install or conda/source-build workaround needed.
!pip install vina meeko gemmi rdkit dimorphite-dl pdbfixer -q

import os, stat, shutil, subprocess, urllib.request, json as _json

# --- Verify OpenBabel actually installed (apt silently no-ops on a failed
# fetch unless checked -- every downstream cell shells out to `obabel`) ---
if not shutil.which('obabel'):
    raise RuntimeError(
        "obabel not found after apt install -- check the apt output above for "
        "fetch errors (e.g. a 404 on a transient dependency) and retry: "
        "!apt-get update -qq && apt-get install -y -q openbabel libboost-all-dev"
    )
obabel_version = subprocess.run(['obabel', '-V'], capture_output=True, text=True).stdout.strip()
print(f'obabel: {obabel_version}')

# --- Verify the pip packages actually import ---
for pkg, import_name in [('vina', 'vina'), ('meeko', 'meeko'), ('gemmi', 'gemmi'), ('rdkit', 'rdkit'),
                          ('dimorphite-dl', 'dimorphite_dl'), ('pdbfixer', 'pdbfixer'), ('openmm', 'openmm')]:
    try:
        __import__(import_name)
        print(f'{pkg} import OK')
    except ImportError as e:
        raise RuntimeError(f'{pkg} failed to import after pip install: {e}')

GNINA_PATH = '/content/gnina' if not HPC_MODE else './gnina'

if not os.path.exists(GNINA_PATH):
    with urllib.request.urlopen('https://api.github.com/repos/gnina/gnina/releases/latest') as resp:
        release = _json.load(resp)
    # Match by prefix, not exact name -- GNINA's release asset naming has
    # changed at least once (was 'gnina', now e.g. 'gnina.cuda12.8.static'
    # as of v1.3.3) and will likely change again as CUDA versions bump.
    matching_assets = [a for a in release['assets'] if a['name'].startswith('gnina')]
    if not matching_assets:
        raise RuntimeError(
            f"No asset starting with 'gnina' found in release {release['tag_name']} "
            f"(assets: {[a['name'] for a in release['assets']]}) -- inspect "
            f"https://github.com/gnina/gnina/releases/latest manually."
        )
    asset = matching_assets[0]
    print(f"Downloading GNINA {release['tag_name']} from {asset['browser_download_url']}")
    urllib.request.urlretrieve(asset['browser_download_url'], GNINA_PATH)
    st = os.stat(GNINA_PATH)
    os.chmod(GNINA_PATH, st.st_mode | stat.S_IEXEC)

!{GNINA_PATH} --version

# --- Extension over the 06_docking.ipynb version: this notebook's later
# modules call literal 'gnina' and 'vina' as subprocess commands (not
# GNINA_PATH), so symlink both onto PATH here rather than editing every
# call site downstream. pip's 'vina' package is a Python binding only --
# fetch the real CLI binary the same way GNINA was fetched above. ---
if not os.path.exists('/usr/local/bin/gnina'):
    subprocess.run(['ln', '-sf', os.path.abspath(GNINA_PATH), '/usr/local/bin/gnina'], check=True)

if not shutil.which('vina'):
    with urllib.request.urlopen('https://api.github.com/repos/ccsb-scripps/AutoDock-Vina/releases/latest') as resp:
        vina_release = _json.load(resp)
    vina_assets = [a for a in vina_release['assets']
                   if 'linux_x86_64' in a['name'].lower() and a['name'].lower().startswith('vina')]
    if not vina_assets:
        raise RuntimeError(
            f"No linux_x86_64 vina asset found in release {vina_release['tag_name']} -- inspect "
            f"https://github.com/ccsb-scripps/AutoDock-Vina/releases/latest manually."
        )
    vina_asset = vina_assets[0]
    print(f"Downloading Vina {vina_release['tag_name']} from {vina_asset['browser_download_url']}")
    urllib.request.urlretrieve(vina_asset['browser_download_url'], '/usr/local/bin/vina')
    st = os.stat('/usr/local/bin/vina')
    os.chmod('/usr/local/bin/vina', st.st_mode | stat.S_IEXEC)

print(f"vina on PATH: {shutil.which('vina')}")
print(f"gnina on PATH: {shutil.which('gnina')}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 51.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 107.0 MB/s eta 0:00:00
obabel: Open Babel 3.1.1 -- Feb  7 2022 -- 06:51:49
vina import OK


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


meeko import OK
gemmi import OK
rdkit import OK
dimorphite-dl import OK
pdbfixer import OK
openmm import OK
gnina v1.3.3 master:6fe1ce2   Built Jun 30 2026.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


vina on PATH: /usr/local/bin/vina
gnina on PATH: /usr/local/bin/gnina


In [3]:
import json
import time
import shutil
import hashlib
import datetime
import platform
import subprocess
import warnings
from xml.etree import ElementTree as ET

import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolTransforms
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import Descriptors, Crippen, Lipinski
from pdbfixer import PDBFixer
from openmm.app import PDBFile
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

print('Imports OK.')
print('Python:', platform.python_version())
for pkg in ('rdkit', 'numpy', 'pandas'):
    try:
        mod = __import__(pkg)
        print(f'{pkg}: {getattr(mod, "__version__", "unknown")}')
    except ImportError:
        print(f'{pkg}: NOT INSTALLED')

# External engine binaries -- not Python packages, checked separately so a
# missing binary fails loudly and early rather than deep inside a docking loop.
for exe in ('vina', 'gnina', 'obabel'):
    found = shutil.which(exe)
    print(f'{exe}: {found or "NOT FOUND ON PATH"}')


Imports OK.
Python: 3.13.15
rdkit: 2025.09.6
numpy: 2.1.3
pandas: 2.2.3
vina: /usr/local/bin/vina
gnina: /usr/local/bin/gnina
obabel: /usr/bin/obabel


## Module A: Receptor Design Matrix

Every receptor unit used in this notebook, locked from a structure audit
done before any code was written (recorded in this project's private notes,
not duplicated here since notebooks must stay self-contained -- every fact
needed to run this notebook is in the config cell below, not referenced
externally).

10 production units across 5 targets. CCR5 is the deliberate exception to
the active/inactive-pair pattern: its allosteric small-molecule pocket
(where maraviroc binds) has no experimentally-solved active-state
counterpart, because the pocket's pharmacological role is specifically to
stabilize the inactive conformation. CCR5 instead uses two independent
allosteric-pocket structures (different antagonists) as a robustness
cross-check, and excludes the orthosteric chemokine-recognition site
entirely from production docking -- that site is a genuine protein-protein
interface (its reference ligands are chemokine
proteins, MIP-1a and RANTES, not small molecules), not a small-molecule
target.

In [4]:
# =============================================================================
# DESIGN MATRIX -- one row per receptor unit. Every PDB ID here was live-
# verified against RCSB directly (species, chains, ligand identity, mutation
# count, fusion-protein presence and, where confirmable, insertion site)
# before being locked in. Do not add a new row without doing the same.
# =============================================================================
DESIGN_MATRIX = [
    # target,   state,               pdb_id,  ref_ligand_code, pocket_type,   species, notes
    ('drd2',    'active',            '6VMS',  '08Y', 'orthosteric', 'human',
     'Gi-complex, bromoergocryptine/bromocriptine (DrugBank DB01200)'),
    ('drd2',    'inactive',          '6CM4',  '8NU', 'orthosteric', 'human',
     'T4L fusion, risperidone (DrugBank DB00734)'),
    ('cb2',     'active',            '6KPF',  'E3R', 'orthosteric', 'human',
     'Gi-complex, AM12033 (not in DrugBank)'),
    ('cb2',     'inactive',          '5ZTY',  '9JU', 'orthosteric', 'human',
     'T4L fusion at ICL3 222-235, 5 thermostabilizing mutations (G78.2.48L/'
     'T127.3.46A/T153.4.45L/R242.6.32E/G304.8.48E), AM10257 (not in DrugBank)'),
    ('adora2a', 'active',            '5G53',  'NEC', 'orthosteric', 'human',
     'mini-Gs complex, NECA (DrugBank DB03719), 1 receptor mutation unidentified'),
    ('adora2a', 'inactive',          '3EML',  'ZMA', 'orthosteric', 'human',
     'T4L fusion, ZM241385 (not in DrugBank)'),
    ('oprm1',   'active',            '8EFO',  '8QY', 'orthosteric', 'human',
     'Gi-complex (scFv16 stabilized), PZM21 (DrugBank DB14030). '
     'Supersedes mouse 6DDF/DAMGO-peptide after audit.'),
    ('oprm1',   'inactive',          '9MQJ',  'A1BNM', 'orthosteric', 'human',
     'Nanobody+Fab stabilized, isoquinuclidine #020_E1 (not in DrugBank), '
     'locally-refined 3.23A (vs globally-deposited 9MQI, same complex). '
     'Supersedes mouse 4DKL/covalent-ligand after audit.'),
    ('ccr5',    'inactive_allosteric_primary',    '4MBS', 'MRV', 'allosteric', 'human',
     'Rubredoxin fusion at ICL3 223-227 (~45A from pocket), maraviroc '
     '(not in this DrugBank export despite being the approved CCR5 drug -- '
     'checked directly, genuinely absent). No active-state structure exists '
     'at this pocket (see module docstring).'),
    ('ccr5',    'inactive_allosteric_comparator', '6AKY', 'A4X', 'allosteric', 'human',
     'Rubredoxin fusion at N-terminus (different site than 4MBS), compound 34 '
     '(not in DrugBank). Robustness cross-check against 4MBS, not primary.'),
]
DESIGN_MATRIX_COLS = ['target', 'state', 'pdb_id', 'ref_ligand_code', 'pocket_type', 'species', 'notes']
design_df = pd.DataFrame(DESIGN_MATRIX, columns=DESIGN_MATRIX_COLS)
assert len(design_df) == 10, f'Expected 10 production units, got {len(design_df)}'

# CCR5's orthosteric/chemokine structures -- excluded from production docking,
# kept here only so Methods can cite exactly what was considered and why it
# was excluded, grounded in checked facts not assumption.
CCR5_EXCLUDED_UNITS = [
    ('ccr5', 'active_orthosteric_excluded', '7F1Q', 'MIP-1a (protein, not small molecule)',
     'Gi-complex, confirmed human, 2.90A cryo-EM. Excluded: genuine chemokine '
     'protein-protein interface, not a small-molecule-docking-suitable pocket.'),
    ('ccr5', 'active_orthosteric_excluded', '7F1R', 'RANTES/CCL5 (protein, not small molecule)',
     'Gi-complex, confirmed human, 3.00A cryo-EM. Same exclusion rationale as 7F1Q.'),
]

# =============================================================================
# SODIUM-ION / CRYSTALLOGRAPHIC WATER RETENTION POLICY -- decided once, applied
# identically to every unit that has a conserved allosteric Na+ site (DRD2,
# ADORA2A), not an ad hoc per-run choice.
# =============================================================================
NA_POLICY_TARGETS = {'drd2', 'adora2a'}
NA_RETENTION_RULE = (
    "Retain a structurally-resolved Na+ ion if it sits in the conserved "
    "allosteric sodium site and does not clash with the docking box. "
    "Strip bulk/ordered waters by default. Retain only waters that directly "
    "bridge the reference ligand or coordinate the sodium site, logged per unit."
)

# =============================================================================
# REDOCKING VALIDATION POLICY
# =============================================================================
MAX_REDOCK_ATTEMPTS = 6          # stop at first pass, not majority-of-N (explicit
                                  # decision, not the default) -- see docstring below
RMSD_PASS_THRESHOLD = 2.0        # Angstrom, heavy-atom RMSD, matches CB2 predecessor convention
BASE_SEED = 42
BASE_EXHAUSTIVENESS = 8          # Vina; matches the CB2 predecessor's proven fixed
                                  # value (06_docking.ipynb used exhaustiveness=8
                                  # flat, no retry escalation at all -- confirmed by
                                  # reading that notebook directly, not assumed).
MAX_EXHAUSTIVENESS = 16          # Cap at 2x base, not unbounded x-attempt growth.
                                  # An earlier version scaled exhaustiveness *
                                  # attempt uncapped (8,16,24,32,40,48) -- on Colab's
                                  # weaker CPU allocation this made a single unit's
                                  # worst case (6 failed attempts) take an hour+,
                                  # threatening to blow the 12h Colab GPU session
                                  # limit across 10 units. Capping at 16 keeps the
                                  # "search harder on retry" idea (still escalates
                                  # once) while bounding worst-case cost per unit.

def redock_seed_and_exhaustiveness(attempt: int):
    '''Each retry varies BOTH the random seed and search exhaustiveness --
    varying only the seed on a stochastic search is defensible, but pairing
    it with "search harder" gives a stronger methodological story for
    Methods than seed-only retries. Exhaustiveness escalates once (8 -> 16)
    then holds at MAX_EXHAUSTIVENESS for remaining attempts, rather than
    growing unbounded with attempt number.'''
    exhaustiveness = min(BASE_EXHAUSTIVENESS * attempt, MAX_EXHAUSTIVENESS)
    return BASE_SEED + attempt, exhaustiveness

# =============================================================================
# GRID BOX DEFINITION -- fixed cube, SAME size for both engines AND every
# unit, centred on the reference ligand's own centroid. Explicitly NOT
# GNINA's --autobox_ligand (ligand-footprint + implicit padding) and
# explicitly NOT a per-ligand/variable box size -- either would make
# consensus reflect mismatched search volumes rather than real scoring
# disagreement, whether that mismatch is between engines or between units.
#
# Raised from 20.0 -> 24.0 A after live redocking validation: 20.0 A
# (inherited unchanged from the CB2 predecessor project) proved too tight
# for drd2_inactive specifically -- failed all 6 attempts at 20.0 A
# (best RMSD 2.37 A), passed on attempt 1 at 24.0 A (RMSD 0.33 A). Kept
# as ONE global fixed size for every unit, not a per-ligand heuristic --
# a bigger fixed size preserves the original rationale; a variable size
# would reopen the exact mismatched-search-volume problem that rationale
# was written to avoid.
# =============================================================================
GRID_BOX_SIZE = 24.0   # Angstrom cube

# =============================================================================
# ENGINE SETTINGS
# =============================================================================
VINA_EXHAUSTIVENESS = 8
VINA_N_POSES = 9
GNINA_N_POSES = 9

# =============================================================================
# CONSENSUS CRITERION -- defined in advance, not post hoc.
# =============================================================================
CONSENSUS_TOP_PERCENT = 20   # a compound "passes" if BOTH engines rank it within
                              # the top X% of that unit's docked candidate set

# =============================================================================
# CANDIDATE SOURCE -- PRIMARY PROSPECTIVE SCREEN (Option A).
#
# Notebook 4 may retain Ki, Ki+IC50 and alternative feature-representation
# screens for sensitivity analysis, but production docking is intentionally
# restricted to the target-specific DEPLOYED classifier applied to the
# FULL activity pool with the COMBINED representation.
#
# Within that provenance-clean primary screen, candidates must additionally
# be dataset-novel and inside the applicability domain. Up to
# N_CANDIDATES_PER_TARGET are then selected by priority_score. Cross-
# representation agreement is retained only as a FULL-pool sensitivity
# annotation and never changes eligibility for docking.
# =============================================================================
N_CANDIDATES_PER_TARGET = 20
CANDIDATE_SOURCE_DESCRIPTION = (
    "Primary prospective DrugBank screen only: deployed FULL-pool + COMBINED-"
    "representation classifier per target; within-AD + dataset-novel hard gates; "
    "top N by priority_score. Cross-representation agreement from the FULL pool "
    "is annotation-only."
)

# =============================================================================
# STRUCTURAL ALERT / DRUG-LIKENESS FILTERS -- applied to docking hits as an
# additional disclosure layer, same PAINS/Brenk/Lipinski/Muegge conventions
# used in notebook 2's curation and notebook 4's screening, not reinvented.
# =============================================================================
APPLY_STRUCTURAL_ALERT_FILTER = True

data_path     = PROJECT_DIR / 'data'
results_path  = PROJECT_DIR / 'ml' / 'results'
screen_path   = results_path / 'drugbank_screening'
protein_dir   = PROJECT_DIR / 'docking' / 'protein'
ligand_dir    = PROJECT_DIR / 'docking' / 'ligands'
logs_dir      = PROJECT_DIR / 'docking' / 'logs'
dockres_dir   = PROJECT_DIR / 'docking' / 'docking_results'
figures_dir   = PROJECT_DIR / 'docking' / 'figures'
for d in (protein_dir, ligand_dir, logs_dir, dockres_dir, figures_dir):
    d.mkdir(parents=True, exist_ok=True)

# Optional shard selector, same convention as notebook 3 -- empty means every
# unit in DESIGN_MATRIX runs in this job.
DOCKING_UNITS = os.environ.get('DOCKING_UNITS', '')
if DOCKING_UNITS:
    _requested = set(DOCKING_UNITS.split(','))
    RUN_UNITS = [row for row in DESIGN_MATRIX if f"{row[0]}_{row[1]}" in _requested]
else:
    RUN_UNITS = DESIGN_MATRIX
IS_SHARDED_RUN = bool(DOCKING_UNITS)
SHARD_TAG = '_' + DOCKING_UNITS.replace(',', '_') if IS_SHARDED_RUN else ''

def shard_name(filename: str) -> str:
    '''Appends the shard tag to a GLOBAL (non-per-unit) output filename so
    concurrent shard jobs do not clobber each other -- identical convention
    to notebook 03.'''
    if not IS_SHARDED_RUN:
        return filename
    stem, _, ext = filename.rpartition('.')
    return f'{stem}{SHARD_TAG}.{ext}' if stem else f'{filename}{SHARD_TAG}'

def log_milestone(unit_key: str, message: str):
    '''Append a timestamped line to a run log that survives a disconnect --
    same purpose as notebooks 3/4's log_milestone, scoped per docking unit.'''
    log_path = logs_dir / f'run_log_05_docking_{unit_key}.txt'
    with open(log_path, 'a') as f:
        f.write(f'[{datetime.datetime.now().isoformat()}] {message}\n')
    return log_path

print(f'Design matrix: {len(DESIGN_MATRIX)} production units, running {len(RUN_UNITS)} this session')
print(f'Sharded run: {IS_SHARDED_RUN}' + (f' ({DOCKING_UNITS})' if IS_SHARDED_RUN else ''))
print(design_df.to_string(index=False))


Design matrix: 10 production units, running 10 this session
Sharded run: False
 target                          state pdb_id ref_ligand_code pocket_type species                                                                                                                                                                                                                                             notes
   drd2                         active   6VMS             08Y orthosteric   human                                                                                                                                                                                    Gi-complex, bromoergocryptine/bromocriptine (DrugBank DB01200)
   drd2                       inactive   6CM4             8NU orthosteric   human                                                                                                                                                                                                

## Module B: Receptor Preparation

Per-unit: download the structure, keep only the receptor chain (drop
G-protein subunits, scFv16, nanobodies, Fab fragments -- none of these are
covalently fused, so chain-level selection removes them cleanly), excise
any covalently-fused stabilizer (T4 lysozyme, rubredoxin) at its confirmed
residue boundary, apply the sodium/water retention policy where relevant,
protonate, and convert to PDBQT.

**Not every fusion boundary is confirmed.** Where the audit could not pin
down the exact insertion residues (DRD2 6CM4, ADORA2A 3EML), this module
flags the unit loudly rather than guessing at a boundary -- automated
excision only proceeds where the boundary is confirmed; unconfirmed units
require a manual coordinate check before their receptor file is trusted.

In [5]:
# =============================================================================
# Per-unit receptor-chain and fusion-excision detail. auth_chain is the
# PUBLIC (author-assigned) chain ID to keep -- the one holding the actual
# receptor sequence, confirmed per-unit during the structure audit.
# fusion_excision_range is (start_resi, end_resi) of a COVALENTLY-FUSED
# stabilizer within that chain, or None if no fusion is present, or
# 'UNCONFIRMED' if a fusion is known to exist but its exact boundary was
# not confirmable from available sources -- these units get flagged, not
# silently guessed at.
# =============================================================================
RECEPTOR_CHAIN_DETAIL = {
    ('drd2', 'active'):     {'auth_chain': 'R', 'fusion_excision_range': None,
                              'strip_hetero': ['GDP'], 'na_policy_applies': True},
    ('drd2', 'inactive'):   {'auth_chain': 'A', 'fusion_excision_range': 'UNCONFIRMED',
                              'strip_hetero': ['OLA', 'PEG'], 'na_policy_applies': True},
    ('cb2', 'active'):      {'auth_chain': 'R', 'fusion_excision_range': None,
                              'strip_hetero': [], 'na_policy_applies': False},
    ('cb2', 'inactive'):    {'auth_chain': 'A', 'fusion_excision_range': (222, 235),
                              'strip_hetero': ['OLC', 'OLA', 'EPE', 'PG4', 'PEG', 'SO4'],
                              'na_policy_applies': False},
    ('adora2a', 'active'):  {'auth_chain': 'A', 'fusion_excision_range': None,
                              'strip_hetero': [], 'na_policy_applies': True,
                              'keep_hetero': ['GDP']},
    ('adora2a', 'inactive'):{'auth_chain': 'A', 'fusion_excision_range': 'UNCONFIRMED',
                              'strip_hetero': ['STE', 'SO4'], 'na_policy_applies': True},
    ('oprm1', 'active'):    {'auth_chain': 'R', 'fusion_excision_range': None,
                              'strip_hetero': ['CLR'], 'na_policy_applies': False},
    ('oprm1', 'inactive'):  {'auth_chain': 'A', 'fusion_excision_range': None,
                              'strip_hetero': [], 'na_policy_applies': False},
    ('ccr5', 'inactive_allosteric_primary'):    {'auth_chain': 'A', 'fusion_excision_range': (223, 227),
                              'strip_hetero': [], 'na_policy_applies': False},
    ('ccr5', 'inactive_allosteric_comparator'): {'auth_chain': 'A', 'fusion_excision_range': 'UNCONFIRMED',
                              'strip_hetero': [], 'na_policy_applies': False,
                              'notes': 'N-terminal rubredoxin fusion confirmed present; exact '
                                       'boundary not confirmed from available sources'},
}

_unconfirmed = [k for k, v in RECEPTOR_CHAIN_DETAIL.items() if v['fusion_excision_range'] == 'UNCONFIRMED']
print(f'{len(_unconfirmed)} unit(s) have a confirmed fusion but an UNCONFIRMED excision boundary:')
for u in _unconfirmed:
    print(f'  {u} -- requires manual mmCIF/coordinate check before automated excision is trusted')


3 unit(s) have a confirmed fusion but an UNCONFIRMED excision boundary:
  ('drd2', 'inactive') -- requires manual mmCIF/coordinate check before automated excision is trusted
  ('adora2a', 'inactive') -- requires manual mmCIF/coordinate check before automated excision is trusted
  ('ccr5', 'inactive_allosteric_comparator') -- requires manual mmCIF/coordinate check before automated excision is trusted


In [6]:
def download_structure(pdb_id: str, out_dir) -> Path:
    '''Downloads the PDB-format coordinate file from RCSB. Uses PDB format
    (not mmCIF) for compatibility with the downstream RDKit/OpenBabel prep
    chain -- this project's existing docking/protein files (CB2's 5ZTY,
    6KPF, copied from the predecessor) are also plain PDB.

    Some newer/larger cryo-EM depositions (confirmed for 9MQJ) are RCSB-
    served as mmCIF ONLY -- no legacy .pdb file exists, curl returns 404
    (checked directly). Falls back to downloading the .cif and converting
    via obabel (already a pipeline dependency) rather than failing the
    unit outright.

    Returns (path, used_cif_fallback) -- NOT just a Path -- so callers can
    disclose the fallback in the prep-status record. (pathlib.Path has
    __slots__ and rejects arbitrary attribute assignment, so this cannot
    be bolted onto the returned Path itself.)'''
    out_path = out_dir / f'{pdb_id}.pdb'
    if out_path.exists():
        return out_path, False
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    result = subprocess.run(['curl', '-sf', '-o', str(out_path), url])
    if result.returncode == 0 and out_path.exists() and out_path.stat().st_size > 0:
        return out_path, False

    print(f'  \u26a0\ufe0f  {pdb_id}: no legacy .pdb at RCSB, falling back to mmCIF conversion')
    cif_path = out_dir / f'{pdb_id}.cif'
    cif_url = f'https://files.rcsb.org/download/{pdb_id}.cif'
    subprocess.run(['curl', '-sf', '-o', str(cif_path), cif_url], check=True)
    assert cif_path.exists() and cif_path.stat().st_size > 0, f'mmCIF download also failed for {pdb_id}'
    subprocess.run(['obabel', str(cif_path), '-O', str(out_path)], check=True, capture_output=True)
    assert out_path.exists() and out_path.stat().st_size > 0, f'mmCIF->PDB conversion failed for {pdb_id}'
    return out_path, True


def extract_receptor_chain(pdb_path: Path, auth_chain: str, out_path: Path,
                            keep_hetero_resnames=None):
    '''Keeps ATOM/HETATM records for the given author chain only, plus any
    explicitly-kept heteroatom residues (reference ligand, retained ions/
    waters per the sodium policy). Everything else -- other chains (G-protein
    subunits, antibody fragments, nanobodies), crystallization additives not
    in the keep list -- is dropped. This does NOT excise a covalent fusion
    within the kept chain; that is a separate step (excise_fusion below),
    since a fused stabilizer shares the same chain ID as the receptor.

    Rebuilds TER/END rather than blindly preserving the original file's
    records -- confirmed live: the original file's TER lines terminate
    OTHER chains that just got filtered out, leaving a stray TER with no
    corresponding ATOM block before it, which crashes OpenMM's PDB parser
    inside PDBFixer (self._current_model is None when it hits that TER).
    SEQRES/HEADER/REMARK-style lines ARE still kept (PDBFixer uses SEQRES
    for missing-residue detection), just reordered to precede the
    coordinate block rather than staying interleaved at their original
    file positions.'''
    keep_hetero_resnames = set(keep_hetero_resnames or [])
    # Allowlist of standard wwPDB header-record prefixes -- position-
    # independent, always safe before the coordinate block. SEQRES is the
    # one PDBFixer actually uses (missing-residue detection); the rest are
    # harmless metadata. Everything NOT in this list (TER, END, MODEL,
    # ENDMDL, CONECT, ANISOU, MASTER, ...) is dropped rather than blocklisted
    # one crash at a time -- confirmed live that both TER and CONECT records
    # crash OpenMM's PDB parser when relocated ahead of any coordinate data
    # (self._current_model is None until the first ATOM/HETATM is seen), and
    # there's no reason to assume those are the only two record types with
    # that same ordering sensitivity.
    _pdb_header_prefixes = (
        'HEADER', 'OBSLTE', 'TITLE', 'SPLIT', 'CAVEAT', 'COMPND', 'SOURCE',
        'KEYWDS', 'EXPDTA', 'NUMMDL', 'MDLTYP', 'AUTHOR', 'REVDAT', 'SPRSDE',
        'JRNL', 'REMARK', 'DBREF', 'SEQADV', 'SEQRES', 'MODRES', 'HET',
        'HETNAM', 'HETSYN', 'FORMUL', 'HELIX', 'SHEET', 'SSBOND', 'LINK',
        'CISPEP', 'SITE', 'CRYST1', 'ORIGX1', 'ORIGX2', 'ORIGX3', 'SCALE1',
        'SCALE2', 'SCALE3', 'MTRIX1', 'MTRIX2', 'MTRIX3',
    )
    header_lines = []
    coord_lines = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                chain_id = line[21]
                resname = line[17:20].strip()
                if chain_id != auth_chain:
                    continue
                if line.startswith('HETATM') and resname not in keep_hetero_resnames and resname != 'HOH':
                    continue
                coord_lines.append(line)
            elif line.startswith(_pdb_header_prefixes):
                header_lines.append(line)
            # else: silently dropped (TER/END/MODEL/ENDMDL/CONECT/ANISOU/
            # MASTER/etc.) -- rebuilt cleanly below instead.
    with open(out_path, 'w') as f:
        f.writelines(header_lines)
        f.writelines(coord_lines)
        if coord_lines:
            f.write('TER\n')
        f.write('END\n')
    return out_path


def excise_fusion(pdb_path: Path, excision_range, out_path: Path):
    '''Removes ATOM records for residues inside the confirmed fusion-protein
    range. Does not attempt to reconnect/cap the receptor sequence across the
    gap -- that is a receptor-prep-time decision (e.g. via Modeller/PyMOL
    loop closure) outside what this notebook automates; flagged in the unit
    manifest for manual follow-up, not silently done.'''
    if excision_range in (None, 'UNCONFIRMED'):
        return pdb_path  # nothing to excise, or excision deliberately deferred
    start, end = excision_range
    kept_lines = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith('ATOM'):
                try:
                    resi = int(line[22:26])
                except ValueError:
                    kept_lines.append(line)
                    continue
                if start <= resi <= end:
                    continue
            kept_lines.append(line)
    with open(out_path, 'w') as f:
        f.writelines(kept_lines)
    return out_path


def protonate_and_convert_pdbqt(pdb_path: Path, out_pdbqt: Path, ph: float = 7.4):
    '''Protonates at physiological pH and converts to PDBQT via OpenBabel.
    OpenBabel is used here (not meeko's receptor prep, which expects a
    pre-protonated, pre-cleaned structure as input) specifically because it
    handles the protonation step in one call -- meeko is used downstream for
    LIGAND prep instead, where its AutoDock4-style atom typing matters more.'''
    subprocess.run(
        ['obabel', str(pdb_path), '-O', str(out_pdbqt), '-p', str(ph), '-xr'],
        check=True, capture_output=True,
    )
    assert out_pdbqt.exists(), f'Protonation/PDBQT conversion failed for {pdb_path}'
    return out_pdbqt


def repair_missing_residues(pdb_path: Path, out_path: Path):
    '''PDBFixer structural completion -- finds and fills missing residues
    and missing atoms (loop gaps common in X-ray/cryo-EM structures,
    especially near flexible loops that can sit close to a binding pocket).
    Deliberately does NOT run PDBFixer's forcefield-based
    addMissingHydrogens or energy minimization -- both require explicit
    forcefield templates for every residue present, and would hard-fail
    on retained non-standard heteroatoms this project's keep_hetero policy
    intentionally preserves (GDP for adora2a_active, Na+ where
    na_policy_applies). Hydrogen placement stays with obabel
    (protonate_and_convert_pdbqt, downstream), which has no such
    template requirement. Applied uniformly to all units, not just the
    ones that failed redocking validation -- no per-unit special-casing.'''
    fixer = PDBFixer(filename=str(pdb_path))
    fixer.findMissingResidues()

    # Drop TERMINAL missing-residue gaps (chain start/end) before filling --
    # confirmed live: PDBFixer's addMissingAtoms() crashes with
    # "'NoneType' object has no attribute '_current_chain'" on 2/10 units
    # when a gap sits at a chain terminus, a known PDBFixer limitation (its
    # internal chain-tracking assumes a preceding resolved residue exists,
    # which isn't true for a truncation at the very start/end of a chain).
    # Internal loop gaps -- the ones actually relevant near a binding
    # pocket -- are unaffected and still get filled; a truncated, floppy
    # terminus far from the pocket doesn't need repair for docking anyway.
    chains = list(fixer.topology.chains())
    terminal_keys = []
    for (chain_idx, res_idx) in list(fixer.missingResidues.keys()):
        chain_len = len(list(chains[chain_idx].residues()))
        if res_idx == 0 or res_idx >= chain_len:
            terminal_keys.append((chain_idx, res_idx))
    for key in terminal_keys:
        del fixer.missingResidues[key]

    n_missing_res = sum(len(v) for v in fixer.missingResidues.values())
    fixer.findMissingAtoms()
    n_missing_atoms = sum(len(v) for v in fixer.missingAtoms.values())
    fixer.addMissingAtoms()
    with open(out_path, 'w') as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
    return out_path, n_missing_res, n_missing_atoms


print('Receptor prep functions defined.')


Receptor prep functions defined.


In [7]:
# =============================================================================
# RECEPTOR PREP MASTER LOOP -- per-unit isolated and checkpointed. A failure
# in one unit is caught, logged, and does not stop the others (per-unit
# isolation, decided before this notebook was built).
# =============================================================================
receptor_prep_status = []

for target, state, pdb_id, ref_ligand_code, pocket_type, species, notes in RUN_UNITS:
    unit_key = f'{target}_{state}'
    detail = RECEPTOR_CHAIN_DETAIL[(target, state)]
    prepared_pdbqt = protein_dir / f'{unit_key}_{pdb_id}_prepared.pdbqt'

    # Compute receptor-prep policy before the cache check so cached and freshly
    # prepared units report identical metadata. This also prevents fresh-prep
    # NameError failures when a cached receptor is absent.
    excision_range = detail['fusion_excision_range']
    receptor_prep_provisional = excision_range == 'UNCONFIRMED'
    keep_hetero = set(detail.get('keep_hetero', []))
    if detail.get('na_policy_applies'):
        keep_hetero.add('NA')  # PDB Chemical Component Dictionary code for Na+

    if prepared_pdbqt.exists():
        print(f'  ⏭️  {unit_key}: receptor checkpoint found, skipping prep')
        receptor_prep_status.append({
            'unit': unit_key, 'target': target, 'state': state, 'pdb_id': pdb_id,
            'auth_chain': detail['auth_chain'], 'fusion_excision_range': str(excision_range),
            'fusion_boundary_confirmed': not receptor_prep_provisional,
            'receptor_prep_provisional': receptor_prep_provisional,
            'kept_heterogens': ';'.join(sorted(keep_hetero)),
            'status': 'cached', 'error': None, 'used_cif_fallback': None,
        })
        continue

    try:
        raw_pdb, used_cif_fallback = download_structure(pdb_id, protein_dir)

        # BUGFIX: this previously unioned ref_ligand_code into the RECEPTOR's
        # own kept heteroatoms -- meaning the receptor file still physically
        # contained the crystallographic ligand, causing every redocking
        # attempt to clash against a copy of itself already baked into the
        # rigid receptor. The reference ligand is extracted separately from
        # raw_pdb/cif directly (Module D), never from this file, so nothing
        # downstream needs it retained here. Receptor must be apo.
        chain_only = protein_dir / f'{unit_key}_{pdb_id}_chain.pdb'
        extract_receptor_chain(raw_pdb, detail['auth_chain'], chain_only, keep_hetero_resnames=keep_hetero)

        if excision_range == 'UNCONFIRMED':
            log_milestone(unit_key, f'WARNING: fusion present but excision boundary unconfirmed '
                                     f'-- proceeding WITHOUT excision, receptor prep is provisional')
            print(f'  ⚠️  {unit_key}: fusion boundary unconfirmed, receptor kept WITH '
                  f'fusion still attached -- flag for manual review before trusting docking results')
            excised = chain_only
        else:
            excised = excise_fusion(chain_only, excision_range, protein_dir / f'{unit_key}_{pdb_id}_excised.pdb')

        repaired_path = protein_dir / f'{unit_key}_{pdb_id}_repaired.pdb'
        repaired, n_missing_res, n_missing_atoms = repair_missing_residues(excised, repaired_path)
        if n_missing_res or n_missing_atoms:
            log_milestone(unit_key, f'PDBFixer: filled {n_missing_res} missing residue(s), '
                                     f'{n_missing_atoms} missing atom(s)')
            print(f'  🔧 {unit_key}: PDBFixer filled {n_missing_res} missing residue(s), '
                  f'{n_missing_atoms} missing atom(s)')

        prepared_pdbqt = protonate_and_convert_pdbqt(repaired, prepared_pdbqt)

        log_milestone(unit_key, f'Receptor prep complete: {pdb_id}, chain {detail["auth_chain"]}, '
                                 f'fusion_excision={excision_range}')
        receptor_prep_status.append({
            'unit': unit_key, 'target': target, 'state': state, 'pdb_id': pdb_id,
            'auth_chain': detail['auth_chain'], 'fusion_excision_range': str(excision_range),
            'fusion_boundary_confirmed': not receptor_prep_provisional,
            'receptor_prep_provisional': receptor_prep_provisional,
            'kept_heterogens': ';'.join(sorted(keep_hetero)),
            'status': 'prepared', 'error': None, 'used_cif_fallback': used_cif_fallback,
        })
        print(f'  ✅ {unit_key}: receptor prepared -> {prepared_pdbqt.name}')
    except Exception as exc:
        log_milestone(unit_key, f'RECEPTOR PREP FAILED: {exc}')
        receptor_prep_status.append({
            'unit': unit_key, 'target': target, 'state': state, 'pdb_id': pdb_id,
            'auth_chain': detail['auth_chain'], 'fusion_excision_range': str(excision_range),
            'fusion_boundary_confirmed': not receptor_prep_provisional,
            'receptor_prep_provisional': receptor_prep_provisional,
            'kept_heterogens': ';'.join(sorted(keep_hetero)),
            'status': 'failed', 'error': str(exc), 'used_cif_fallback': None,
        })
        print(f'  ❌ {unit_key}: receptor prep FAILED -- {exc} -- other units unaffected, continuing')
        continue

receptor_prep_df = pd.DataFrame(receptor_prep_status)
receptor_prep_path = dockres_dir / shard_name('receptor_prep_status.csv')
receptor_prep_df.to_csv(receptor_prep_path, index=False)
print(f'\nReceptor prep status saved: {receptor_prep_path}')
if len(receptor_prep_df):
    display_cols = ['unit', 'status', 'receptor_prep_provisional', 'fusion_boundary_confirmed']
    print(receptor_prep_df[[c for c in display_cols if c in receptor_prep_df.columns]].to_string(index=False))
else:
    print('(no units processed)')



  ⏭️  drd2_active: receptor checkpoint found, skipping prep
  ⏭️  drd2_inactive: receptor checkpoint found, skipping prep
  ⏭️  cb2_active: receptor checkpoint found, skipping prep
  ⏭️  cb2_inactive: receptor checkpoint found, skipping prep
  ⏭️  adora2a_active: receptor checkpoint found, skipping prep
  ⏭️  adora2a_inactive: receptor checkpoint found, skipping prep
  ⏭️  oprm1_active: receptor checkpoint found, skipping prep
  ⏭️  oprm1_inactive: receptor checkpoint found, skipping prep
  ⏭️  ccr5_inactive_allosteric_primary: receptor checkpoint found, skipping prep
  ⏭️  ccr5_inactive_allosteric_comparator: receptor checkpoint found, skipping prep

Receptor prep status saved: /content/drive/My Drive/gpcr_benchmark/docking/docking_results/receptor_prep_status.csv
                               unit status  receptor_prep_provisional  fusion_boundary_confirmed
                        drd2_active cached                      False                       True
                      drd2_ina

## Module C: Docking Candidate Set

Loads notebook 4's **primary prospective DrugBank output** and selects up to
`N_CANDIDATES_PER_TARGET` compounds per target by `priority_score`.

Production docking is restricted to candidates originating from the
**deployed full-pool, combined-representation classifier** for each target.
Dataset novelty and applicability-domain membership are retained as hard gates.
The notebook explicitly validates this provenance before selecting candidates.

Cross-representation agreement is retained only as a sensitivity annotation,
and only the **full-pool** consensus rows are allowed to annotate the primary
candidate set. It is not an inclusion criterion.


In [11]:
primary_candidates_path = screen_path / 'drugbank_primary_ranked_candidates.csv'
assert primary_candidates_path.exists(), (
    f'notebook 4 output not found at {primary_candidates_path} -- run '
    f'04_drugbank_screening.ipynb before this notebook.'
)

primary_candidates_df = pd.read_csv(primary_candidates_path)

# -------------------------------------------------------------------------
# PROVENANCE GUARDRAILS
#
# Notebook 5 must never silently fall back to the older mixed-pool candidate
# union. The primary source must contain only:
#   - FULL activity-pool predictions
#   - COMBINED feature representation
#   - one deployed classifier per target
#
# Notebook 4 has already resolved and applied the deployed classifier.
# Therefore Notebook 5 validates the provenance carried by the actual
# primary screening artifact rather than independently re-selecting a model
# from best_algorithm_by_combination.csv.
# -------------------------------------------------------------------------
required_candidate_cols = {
    'target',
    'activity_pool',
    'feature_representation',
    'algorithm',
    'within_applicability_domain',
    'already_in_training_set',
    'priority_score',
    'global_compound_id'
}

missing_cols = required_candidate_cols - set(primary_candidates_df.columns)
assert not missing_cols, (
    'drugbank_primary_ranked_candidates.csv is missing required provenance '
    f'columns: {sorted(missing_cols)}'
)

# -------------------------------------------------------------------------
# Hard provenance checks
# -------------------------------------------------------------------------
bad_pool = primary_candidates_df[
    primary_candidates_df['activity_pool'] != 'full'
]

bad_rep = primary_candidates_df[
    primary_candidates_df['feature_representation'] != 'combined'
]

assert len(bad_pool) == 0, (
    f'PROVENANCE FAILURE: primary DrugBank file contains '
    f'{len(bad_pool)} non-full row(s). '
    'Re-run patched notebook 4 before docking.'
)

assert len(bad_rep) == 0, (
    f'PROVENANCE FAILURE: primary DrugBank file contains '
    f'{len(bad_rep)} non-combined row(s). '
    'Re-run patched notebook 4 before docking.'
)

# -------------------------------------------------------------------------
# Resolve deployed classifier directly from the corrected Notebook 4
# primary-screen artifact.
#
# Each target must have exactly ONE algorithm represented in the primary
# candidate file. Multiple algorithms for the same target would indicate
# that mixed-model provenance has leaked back into the production screen.
# -------------------------------------------------------------------------
deployed_algorithm_by_target = {}
provenance_errors = []

for target, grp in primary_candidates_df.groupby('target'):

    observed_algorithms = sorted(
        grp['algorithm']
        .dropna()
        .astype(str)
        .unique()
    )

    if len(observed_algorithms) == 0:
        provenance_errors.append(
            f'{target}: no classifier provenance recorded'
        )
        continue

    if len(observed_algorithms) > 1:
        provenance_errors.append(
            f'{target}: multiple classifiers observed '
            f'{observed_algorithms}'
        )
        continue

    deployed_algorithm_by_target[target] = observed_algorithms[0]

assert not provenance_errors, (
    'PROVENANCE FAILURE in primary DrugBank candidates:\n  - '
    + '\n  - '.join(provenance_errors)
)

print('Primary DrugBank provenance verified from Notebook 4 output:')
for target in sorted(deployed_algorithm_by_target):
    print(
        f'  {target}: full / combined / '
        f'{deployed_algorithm_by_target[target]}'
    )

# -------------------------------------------------------------------------
# Sensitivity-only annotation
#
# cross_representation_consensus.csv contains rows across activity pools.
# Only FULL-pool rows are allowed to annotate production candidates.
#
# This annotation is NOT an eligibility criterion for docking.
# -------------------------------------------------------------------------
consensus_path = screen_path / 'cross_representation_consensus.csv'

consensus_df = (
    pd.read_csv(consensus_path)
    if consensus_path.exists()
    else None
)

if consensus_df is not None:

    if 'activity_pool' in consensus_df.columns:
        consensus_df = consensus_df[
            consensus_df['activity_pool'] == 'full'
        ].copy()

    print(
        f'Loaded full-pool cross-representation consensus annotations: '
        f'{len(consensus_df)} row(s)'
    )
else:
    print(
        'cross_representation_consensus.csv not found -- '
        'continuing without consensus annotation.'
    )

# -------------------------------------------------------------------------
# Select docking candidates
# -------------------------------------------------------------------------
docking_candidates_frames = []
docking_candidate_selection_records = []

for target in design_df['target'].unique():

    expected_algorithm = deployed_algorithm_by_target.get(target)

    # A target can legitimately have zero primary novel candidates
    if expected_algorithm is None:
        docking_candidate_selection_records.append({
            'target': target,
            'primary_source_activity_pool': 'full',
            'primary_source_feature_representation': 'combined',
            'deployed_classifier': None,
            'available_rows_after_hard_gates': 0,
            'unique_compounds_after_deduplication': 0,
            'duplicate_rows_removed': 0,
            'selected_for_docking': 0,
            'target_candidate_cap': int(N_CANDIDATES_PER_TARGET),
            'selected_unanimous_consensus_full_pool': 0,
            'underfilled_target': True,
        })

        print(
            f'{target}: 0 candidates selected for docking '
            f'(no primary candidate rows present)'
        )
        continue

    sub = primary_candidates_df[
        (primary_candidates_df['target'] == target)
        & (primary_candidates_df['activity_pool'] == 'full')
        & (primary_candidates_df['feature_representation'] == 'combined')
        & (primary_candidates_df['algorithm'] == expected_algorithm)
        & (primary_candidates_df['within_applicability_domain'] == True)
        & (~primary_candidates_df['already_in_training_set'])
    ].copy()

    n_available_rows = len(sub)

    # -------------------------------------------------------------
    # Full-pool cross-representation consensus annotation only
    # -------------------------------------------------------------
    if consensus_df is not None and len(sub):

        unanimous_ids = set(
            consensus_df[
                (consensus_df['target'] == target)
                & (
                    consensus_df[
                        'n_representations_high_confidence'
                    ] == 3
                )
                & (
                    ~consensus_df[
                        'already_in_training_set'
                    ]
                )
            ]['global_compound_id']
        )

        sub['is_unanimous_consensus'] = (
            sub['global_compound_id'].isin(unanimous_ids)
        )

    else:
        sub['is_unanimous_consensus'] = False

    # -------------------------------------------------------------
    # Defensive deduplication
    #
    # Under the corrected primary screen there should normally be
    # one row per target/compound. Keep this guardrail so accidental
    # duplicated upstream rows cannot consume multiple top-N slots.
    # -------------------------------------------------------------
    sub = sub.sort_values(
        'priority_score',
        ascending=False
    )

    sub_unique = sub.drop_duplicates(
        subset=['target', 'global_compound_id'],
        keep='first'
    )

    n_duplicate_rows_removed = (
        n_available_rows - len(sub_unique)
    )

    # -------------------------------------------------------------
    # Select top-N candidates for docking
    # -------------------------------------------------------------
    top_n = sub_unique.head(
        N_CANDIDATES_PER_TARGET
    ).copy()

    docking_candidates_frames.append(top_n)

    n_unanimous = (
        int(top_n['is_unanimous_consensus'].sum())
        if len(top_n)
        else 0
    )

    docking_candidate_selection_records.append({
        'target': target,
        'primary_source_activity_pool': 'full',
        'primary_source_feature_representation': 'combined',
        'deployed_classifier': expected_algorithm,
        'available_rows_after_hard_gates': int(n_available_rows),
        'unique_compounds_after_deduplication': int(len(sub_unique)),
        'duplicate_rows_removed': int(n_duplicate_rows_removed),
        'selected_for_docking': int(len(top_n)),
        'target_candidate_cap': int(N_CANDIDATES_PER_TARGET),
        'selected_unanimous_consensus_full_pool': int(n_unanimous),
        'underfilled_target': bool(
            len(top_n) < N_CANDIDATES_PER_TARGET
        ),
    })

    print(
        f'{target}: {len(top_n)} candidates selected for docking from '
        f'FULL / combined / {expected_algorithm} '
        f'({n_unanimous} full-pool cross-representation unanimous; '
        f'removed {n_duplicate_rows_removed} duplicate row(s))'
    )

# -------------------------------------------------------------------------
# Combine selected candidates
# -------------------------------------------------------------------------
docking_candidates_df = (
    pd.concat(
        docking_candidates_frames,
        ignore_index=True
    )
    if docking_candidates_frames
    else pd.DataFrame()
)

# -------------------------------------------------------------------------
# Final production invariant
#
# Every candidate reaching docking must still carry the corrected
# Option-A provenance.
# -------------------------------------------------------------------------
if len(docking_candidates_df):

    assert docking_candidates_df[
        'activity_pool'
    ].eq('full').all(), (
        'PROVENANCE FAILURE: non-full candidate leaked into docking set.'
    )

    assert docking_candidates_df[
        'feature_representation'
    ].eq('combined').all(), (
        'PROVENANCE FAILURE: non-combined candidate leaked into docking set.'
    )

    for target, grp in docking_candidates_df.groupby('target'):

        expected_algorithm = (
            deployed_algorithm_by_target[target]
        )

        assert grp['algorithm'].eq(
            expected_algorithm
        ).all(), (
            f'PROVENANCE FAILURE: {target} contains '
            f'non-primary classifier rows.'
        )

# -------------------------------------------------------------------------
# Save corrected docking candidate set
# -------------------------------------------------------------------------
docking_candidates_path = (
    dockres_dir
    / shard_name('docking_candidates.csv')
)

docking_candidates_df.to_csv(
    docking_candidates_path,
    index=False
)

candidate_selection_summary_df = pd.DataFrame(
    docking_candidate_selection_records
)

docking_candidate_selection_summary_path = (
    dockres_dir
    / shard_name(
        'docking_candidate_selection_summary.csv'
    )
)

candidate_selection_summary_df.to_csv(
    docking_candidate_selection_summary_path,
    index=False
)

print(
    f'\nTotal docking candidates: '
    f'{len(docking_candidates_df)} '
    f'-> {docking_candidates_path}'
)

print(
    f'Candidate-selection summary: '
    f'{docking_candidate_selection_summary_path}'
)

print(
    candidate_selection_summary_df.to_string(
        index=False
    )
    if len(candidate_selection_summary_df)
    else '(no candidates selected)'
)

# -------------------------------------------------------------------------
# Preview reuse of existing docking checkpoints
#
# This does NOT perform docking. It only determines how many of the
# corrected candidate × receptor-unit pairs already have successful
# Vina/GNINA results available.
# -------------------------------------------------------------------------
for engine_name, ckpt_name in [
    ('Vina', 'vina_docking_results.csv'),
    ('GNINA', 'gnina_docking_results.csv'),
]:

    ckpt_path = (
        dockres_dir
        / shard_name(ckpt_name)
    )

    if (
        not ckpt_path.exists()
        or not len(docking_candidates_df)
    ):
        continue

    prev = pd.read_csv(ckpt_path)

    if 'status' in prev.columns:
        prev = prev[
            prev['status'].eq('ok')
        ].copy()

    required_pairs = set()

    for target, grp in docking_candidates_df.groupby(
        'target'
    ):

        units = [
            f'{r[0]}_{r[1]}'
            for r in RUN_UNITS
            if r[0] == target
        ]

        for unit in units:
            for cid in grp[
                'global_compound_id'
            ]:
                required_pairs.add(
                    (unit, cid)
                )

    available_pairs = set(
        zip(
            prev['unit'],
            prev['global_compound_id']
        )
    )

    reusable = (
        required_pairs
        & available_pairs
    )

    missing = (
        required_pairs
        - available_pairs
    )

    print(
        f'{engine_name} checkpoint reuse preview: '
        f'{len(reusable)}/{len(required_pairs)} '
        f'required unit-compound pairs reusable; '
        f'{len(missing)} require docking.'
    )

Primary DrugBank provenance verified from Notebook 4 output:
  adora2a: full / combined / XGBoost
  cb2: full / combined / LightGBM
  drd2: full / combined / XGBoost
  oprm1: full / combined / XGBoost
Loaded full-pool cross-representation consensus annotations: 1723 row(s)
drd2: 20 candidates selected for docking from FULL / combined / XGBoost (3 full-pool cross-representation unanimous; removed 0 duplicate row(s))
cb2: 2 candidates selected for docking from FULL / combined / LightGBM (1 full-pool cross-representation unanimous; removed 0 duplicate row(s))
adora2a: 2 candidates selected for docking from FULL / combined / XGBoost (0 full-pool cross-representation unanimous; removed 0 duplicate row(s))
oprm1: 20 candidates selected for docking from FULL / combined / XGBoost (4 full-pool cross-representation unanimous; removed 0 duplicate row(s))
ccr5: 0 candidates selected for docking (no primary candidate rows present)

Total docking candidates: 44 -> /content/drive/My Drive/gpcr_benchm

## Module D: Ligand Preparation

Two ligand sets get prepared: (1) each unit's reference ligand, extracted
from its own receptor structure, used for redocking validation, and
(2) the docking candidate set from Module C, prepared once and reused
across every unit (structure doesn't depend on the receptor). Both use
`meeko` for AutoDock-style atom typing and PDBQT generation, matching the
`meeko==0.7.1` pin already recorded in this project's environment.

In [12]:
# GNINA/smina only accept standard AutoDock atom types. Meeko can write
# macrocycle glue types such as CG0 in PDBQT files; those are valid for some
# AutoDock-family workflows but GNINA rejects them at parse time. Validate both
# freshly prepared and cached ligands before any docking engine consumes them.
GNINA_COMPATIBLE_PDBQT_TYPES = {
    'H', 'HD', 'HS', 'C', 'A', 'N', 'NA', 'NS', 'OA', 'OS', 'F',
    'Mg', 'MG', 'P', 'SA', 'S', 'Cl', 'CL', 'Ca', 'CA', 'Mn', 'MN',
    'Fe', 'FE', 'Zn', 'ZN', 'Br', 'BR', 'I',
}


def invalid_pdbqt_atom_types(pdbqt_path: Path):
    bad = []
    if not Path(pdbqt_path).exists():
        return ['missing_file']
    with open(pdbqt_path) as f:
        for line_no, line in enumerate(f, start=1):
            if line.startswith(('ATOM', 'HETATM')):
                parts = line.split()
                if not parts:
                    continue
                atom_type = parts[-1]
                if atom_type not in GNINA_COMPATIBLE_PDBQT_TYPES:
                    bad.append(f'{atom_type}@line{line_no}')
    return bad


def pdbqt_is_engine_compatible(pdbqt_path: Path):
    return len(invalid_pdbqt_atom_types(pdbqt_path)) == 0


def cached_ligand_or_none(pdbqt_path: Path):
    if not Path(pdbqt_path).exists():
        return None, 'missing'
    bad = invalid_pdbqt_atom_types(pdbqt_path)
    if bad:
        try:
            Path(pdbqt_path).unlink()
        except FileNotFoundError:
            pass
        return None, 'stale_invalid_atom_types:' + ';'.join(bad[:5])
    return Path(pdbqt_path), 'cached'


def prepare_ligand_pdbqt(mol: Chem.Mol, out_pdbqt: Path, ph: float = 7.4):
    '''Standardizes, protonates at physiological pH via Dimorphite-DL (not
    RDKit's default -- Dimorphite-DL enumerates ionization states using
    empirical pKa data, RDKit alone leaves whatever charge state was in the
    input SMILES), embeds 3D, and converts to PDBQT via meeko. Returns None
    (not an exception) on failure so a single bad candidate does not abort
    the whole ligand-prep loop -- caller is responsible for checking the
    return value.

    NOTE: this pH-aware protonation step applies ONLY to candidate ligands
    (built fresh from SMILES here). Reference ligands (Module D, extracted
    from a crystallographic PDB fragment for redocking validation) are
    intentionally NOT re-protonated this way -- doing so would mean
    discarding the experimental 3D pose to round-trip through SMILES, which
    defeats the purpose of redocking against a known reference geometry.
    That is a disclosed, deliberate asymmetry, not an oversight.

    Dimorphite-DL protonation is a DIFFERENT axis from tautomer handling --
    tautomers (proton placement within the same formula/charge) are already
    canonicalised once, upstream, in notebook 4's clean_structure() via
    RDKit's TautomerEnumerator. This function does not touch tautomers at
    all; it only adjusts ionization state (proton count) at the given pH.'''
    try:
        from dimorphite_dl import protonate_smiles

        mol = rdMolStandardize.Cleanup(mol)
        mol = rdMolStandardize.FragmentParent(mol)
        smiles = Chem.MolToSmiles(mol)
        protonated_variants = protonate_smiles(smiles, ph_min=ph, ph_max=ph, precision=0.5)
        if protonated_variants:
            # A single pH point (ph_min == ph_max) should converge to the
            # dominant state in almost all cases; if Dimorphite-DL still
            # returns >1 variant, take the first (its own most-probable
            # ordering) rather than silently picking an arbitrary one.
            mol = Chem.MolFromSmiles(protonated_variants[0])
        # else: Dimorphite-DL found nothing to protonate (e.g. no ionizable
        # groups) -- fall through and use the standardized, unprotonated mol.

        mol = Chem.AddHs(mol)
        if AllChem.EmbedMolecule(mol, randomSeed=BASE_SEED) != 0:
            return None
        AllChem.MMFFOptimizeMolecule(mol)

        sdf_path = out_pdbqt.with_suffix('.sdf')
        writer = Chem.SDWriter(str(sdf_path))
        writer.write(mol)
        writer.close()

        # --rigid_macrocycles: without this, meeko can break open large/
        # flexible ring systems and mark the break points with glue-atom
        # types (e.g. 'CG0') that only macrocycle-aware engines (AutoDock-
        # GPU) understand. Confirmed live: vanilla GNINA/Vina reject these
        # outright -- "CG0 is not a valid AutoDock type" -- causing a real,
        # ligand-specific docking failure downstream, not a ligand-prep
        # failure (meeko itself succeeds and writes a PDBQT; the writer
        # just isn't compatible with the engines this project uses).
        subprocess.run(
            ['mk_prepare_ligand.py', '-i', str(sdf_path), '-o', str(out_pdbqt), '--rigid_macrocycles'],
            check=True, capture_output=True,
        )
        if not out_pdbqt.exists():
            return None
        bad_types = invalid_pdbqt_atom_types(out_pdbqt)
        if bad_types:
            print(f'  ⚠️  {out_pdbqt.name}: incompatible PDBQT atom types after prep: {bad_types[:5]}')
            try:
                out_pdbqt.unlink()
            except FileNotFoundError:
                pass
            return None
        return out_pdbqt
    except Exception:
        return None


def _extract_ligand_from_cif(cif_path: Path, resname: str):
    '''Pulls one residue's HETATM records straight from the original mmCIF
    file via gemmi, bypassing the lossy obabel mmCIF->PDB conversion used
    for the full receptor (Module B). That conversion silently drops ALL
    heteroatoms for at least one unit (confirmed for 9MQJ: zero HETATM lines
    survived in the converted .pdb) -- likely tied to the same root cause
    that made RCSB serve mmCIF-only in the first place (some newer ligand
    codes, e.g. A1BNM, are 5 characters, longer than legacy PDB's 3-character
    residue-name field; obabel does not attempt a lossy rename, it just
    drops the record).

    Writes a minimal, correctly column-aligned HETATM-only PDB fragment.
    Uses a truncated placeholder resname (first 3 chars) in the written
    file purely to keep column alignment legal for obabel's PDB reader --
    downstream code (RMSD, docking) only uses atom coordinates, never this
    label, so truncation here loses no real information.

    Returns the written Path, or None if no matching residue was found.
    '''
    import gemmi
    st = gemmi.read_structure(str(cif_path))
    match = None
    for chain in st[0]:
        for res in chain:
            if res.name == resname:
                match = res
                break
        if match is not None:
            break
    if match is None:
        return None

    safe_resname = resname[:3]
    lines = []
    for i, atom in enumerate(match, start=1):
        element = atom.element.name.upper()
        lines.append(
            f'HETATM{i:>5} {atom.name:<4} {safe_resname:>3} A{1:>4}    '
            f'{atom.pos.x:8.3f}{atom.pos.y:8.3f}{atom.pos.z:8.3f}'
            f'{atom.occ:6.2f}{atom.b_iso:6.2f}          {element:>2}\n'
        )
    out_path = cif_path.with_name(f'{cif_path.stem}_{resname}_from_cif.pdb')
    with open(out_path, 'w') as f:
        f.writelines(lines)
    return out_path


def _dedup_first_copy(lines, chain_idx=21, resseq_slice=(22, 26)):
    '''Some PDB entries contain multiple copies of the same ligand code --
    e.g. 2 copies of the full receptor complex in one asymmetric unit --
    so grabbing every HETATM line matching a resname regardless of chain/
    copy produces a 2-fragment "molecule" that RDKit/meeko correctly refuse
    (confirmed live: "RDKit molecule has 2 fragments. Must have 1." on 3
    units). Keeps only the first contiguous (chain, resSeq) group
    encountered -- matches the same "first match wins" policy already used
    in _extract_ligand_from_cif for the mmCIF path.'''
    if not lines:
        return lines
    first_key = (lines[0][chain_idx], lines[0][resseq_slice[0]:resseq_slice[1]])
    return [l for l in lines if (l[chain_idx], l[resseq_slice[0]:resseq_slice[1]]) == first_key]


def _correct_bond_orders(raw_pdb_fragment: Path, resname: str, out_dir: Path):
    '''PDB format doesn't reliably encode ligand bond connectivity --
    RDKit's MolFromPDBFile infers bonds from atomic distances alone, which
    can silently get bond order/aromaticity wrong (confirmed live: ZM241385
    failed RDKit sanitization with "Explicit valence for atom # 10 C, 5, is
    greater than permitted" when read without correction). Corrects against
    RCSB's ideal/template SDF (Chemical Component Dictionary) via
    AssignBondOrdersFromTemplate -- the same approach the CB2 predecessor
    project's 06_docking.ipynb already used and proved necessary for
    exactly this reason. Falls back to the uncorrected mol (rather than
    hard-failing the whole ligand) if the template fetch/assignment fails,
    e.g. for a ligand code that has no RCSB ideal-SDF entry.'''
    mol = Chem.MolFromPDBFile(str(raw_pdb_fragment), removeHs=False)
    if mol is None:
        return None
    try:
        ideal_url = f'https://files.rcsb.org/ligands/download/{resname.upper()}_ideal.sdf'
        ideal_sdf_path = out_dir / f'{resname.upper()}_ideal.sdf'
        subprocess.run(['curl', '-sf', '-o', str(ideal_sdf_path), ideal_url], check=True)
        ideal_mol = Chem.MolFromMolFile(str(ideal_sdf_path), removeHs=True)
        if ideal_mol is not None:
            mol = AllChem.AssignBondOrdersFromTemplate(ideal_mol, mol)
    except Exception as exc:
        print(f'  \u26a0\ufe0f  bond-order correction failed for {resname}: {exc} -- using uncorrected geometry')
    mol = Chem.AddHs(mol, addCoords=True)
    corrected_sdf = out_dir / f'{resname}_corrected.sdf'
    with Chem.SDWriter(str(corrected_sdf)) as w:
        w.write(mol)
    return corrected_sdf


# ---- Reference ligands (one per unit, extracted from the receptor structure
# used for redocking validation) ----
reference_ligand_status = []
for target, state, pdb_id, ref_ligand_code, pocket_type, species, notes in RUN_UNITS:
    unit_key = f'{target}_{state}'
    raw_pdb = protein_dir / f'{pdb_id}.pdb'
    cif_path = protein_dir / f'{pdb_id}.cif'
    ref_pdbqt = ligand_dir / f'{unit_key}_{pdb_id}_{ref_ligand_code}_reference.pdbqt'
    if ref_pdbqt.exists():
        bad_types = invalid_pdbqt_atom_types(ref_pdbqt)
        if bad_types:
            reference_ligand_status.append({'unit': unit_key, 'status': 'failed_cached_invalid_atom_types',
                                            'error': ';'.join(bad_types[:5])})
            print(f'  ❌ {unit_key}: cached reference ligand has GNINA-incompatible atom types: {bad_types[:5]}')
        else:
            reference_ligand_status.append({'unit': unit_key, 'status': 'cached'})
        continue
    if not raw_pdb.exists() and not cif_path.exists():
        reference_ligand_status.append({'unit': unit_key, 'status': 'skipped_no_receptor'})
        continue
    try:
        ref_pdb_path = ligand_dir / f'{unit_key}_{pdb_id}_{ref_ligand_code}_raw.pdb'
        source = 'legacy_pdb'
        ref_lines = []
        if raw_pdb.exists():
            ref_lines_all = [l for l in open(raw_pdb) if l.startswith('HETATM') and l[17:20].strip() == ref_ligand_code]
            ref_lines = _dedup_first_copy(ref_lines_all)
        if not ref_lines and cif_path.exists():
            # Either no legacy .pdb existed at all, or it existed but the
            # ligand isn't in it (confirmed case: mmCIF-fallback conversion
            # dropping HETATM records entirely) -- fall back to the
            # original mmCIF, which always has the full, untruncated record.
            cif_extract = _extract_ligand_from_cif(cif_path, ref_ligand_code)
            if cif_extract is None:
                raise ValueError(f'ligand code {ref_ligand_code} not found in {pdb_id}.pdb OR {pdb_id}.cif')
            ref_pdb_path = cif_extract
            source = 'mmcif_direct'
        elif ref_lines:
            with open(ref_pdb_path, 'w') as f:
                f.writelines(ref_lines)
        else:
            raise ValueError(f'ligand code {ref_ligand_code} not found in {pdb_id}.pdb HETATM records (no .cif available as fallback)')

        # Bond-order correction (RCSB ideal template) THEN meeko for the
        # PDBQT write -- meeko/RDKit enforce real valence/fragment-count
        # rules that a bare obabel PDBQT write never checked, which is why
        # this only surfaced now: 3 units had undetected 2-copy contamination,
        # 1 had an undetected bad bond-order guess. Both are real data-
        # quality problems, not meeko being pickier than necessary.
        corrected_sdf = _correct_bond_orders(ref_pdb_path, ref_ligand_code, ligand_dir)
        if corrected_sdf is None:
            raise ValueError(f'could not parse extracted ligand fragment for {ref_ligand_code}')
        protonated_sdf = ligand_dir / f'{unit_key}_{pdb_id}_{ref_ligand_code}_protonated.sdf'
        subprocess.run(['obabel', str(corrected_sdf), '-O', str(protonated_sdf), '-p', '7.4'],
                       check=True, capture_output=True)
        subprocess.run(['mk_prepare_ligand.py', '-i', str(protonated_sdf), '-o', str(ref_pdbqt)],
                       check=True, capture_output=True)
        bad_types = invalid_pdbqt_atom_types(ref_pdbqt)
        if bad_types:
            raise ValueError(f'reference ligand PDBQT has GNINA-incompatible atom types: {bad_types[:5]}')
        reference_ligand_status.append({'unit': unit_key, 'status': 'prepared', 'source': source})
    except Exception as exc:
        reference_ligand_status.append({'unit': unit_key, 'status': 'failed', 'error': str(exc)})
        print(f'  \u274c {unit_key}: reference ligand extraction failed -- {exc}')

print(pd.DataFrame(reference_ligand_status).to_string(index=False))

# ---- Docking candidates (prepared once, target-agnostic) ----
candidate_ligand_status = []
for _, row in docking_candidates_df.iterrows():
    cid = row['global_compound_id']
    out_pdbqt = ligand_dir / f'candidate_{cid}.pdbqt'
    cached, cache_status = cached_ligand_or_none(out_pdbqt)
    if cached is not None:
        candidate_ligand_status.append({'global_compound_id': cid, 'status': cache_status})
        continue
    if cache_status.startswith('stale_invalid_atom_types'):
        print(f'  ♻️  {out_pdbqt.name}: {cache_status}; regenerating')
    mol = Chem.MolFromSmiles(row['clean_smiles'])
    if mol is None:
        candidate_ligand_status.append({'global_compound_id': cid, 'status': 'failed_parse'})
        continue
    result = prepare_ligand_pdbqt(mol, out_pdbqt)
    candidate_ligand_status.append({
        'global_compound_id': cid,
        'status': 'prepared' if result else ('failed_prep_after_invalid_cache' if cache_status.startswith('stale_invalid_atom_types') else 'failed_prep'),
    })

candidate_ligand_df = pd.DataFrame(candidate_ligand_status)
n_ok = (candidate_ligand_df['status'].isin(['prepared', 'cached'])).sum()
print(f'\nCandidate ligands ready: {n_ok} / {len(candidate_ligand_df)}')
candidate_ligand_path = dockres_dir / shard_name('candidate_ligand_prep_status.csv')
candidate_ligand_df.to_csv(candidate_ligand_path, index=False)


                               unit status
                        drd2_active cached
                      drd2_inactive cached
                         cb2_active cached
                       cb2_inactive cached
                     adora2a_active cached
                   adora2a_inactive cached
                       oprm1_active cached
                     oprm1_inactive cached
   ccr5_inactive_allosteric_primary cached
ccr5_inactive_allosteric_comparator cached

Candidate ligands ready: 44 / 44


## Module E: Grid Box Definition

One box per unit, centred on that unit's own reference ligand centroid,
fixed `GRID_BOX_SIZE` cube -- identical box for both Vina and GNINA. Not
GNINA's `--autobox_ligand` (ligand-footprint + implicit padding): that
searches a different volume than Vina's fixed cube, and consensus between
engines needs to reflect real scoring disagreement, not differently-sized
search spaces.

In [13]:
def ligand_centroid_from_pdbqt(pdbqt_path: Path):
    '''Mean heavy-atom coordinate from a PDBQT file -- used as the docking
    box centre. PDBQT ATOM/HETATM columns are fixed-width, same layout as PDB.'''
    coords = []
    with open(pdbqt_path) as f:
        for line in f:
            if line.startswith(('ATOM', 'HETATM')):
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                coords.append((x, y, z))
    coords = np.array(coords)
    return coords.mean(axis=0)


box_definitions = {}
for target, state, pdb_id, ref_ligand_code, pocket_type, species, notes in RUN_UNITS:
    unit_key = f'{target}_{state}'
    ref_pdbqt = ligand_dir / f'{unit_key}_{pdb_id}_{ref_ligand_code}_reference.pdbqt'
    if not ref_pdbqt.exists():
        print(f'  \u26a0\ufe0f  {unit_key}: no reference ligand PDBQT, box undefined, skipping')
        continue
    center = ligand_centroid_from_pdbqt(ref_pdbqt)
    box_definitions[unit_key] = {
        'center_x': float(center[0]), 'center_y': float(center[1]), 'center_z': float(center[2]),
        'size_x': GRID_BOX_SIZE, 'size_y': GRID_BOX_SIZE, 'size_z': GRID_BOX_SIZE,
    }
    print(f'{unit_key}: box center = ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f}), '
          f'size = {GRID_BOX_SIZE} A cube')

box_df = pd.DataFrame.from_dict(box_definitions, orient='index').reset_index().rename(columns={'index': 'unit'})
box_path = dockres_dir / shard_name('grid_box_definitions.csv')
box_df.to_csv(box_path, index=False)
print(f'\nGrid box definitions saved: {box_path}')


drd2_active: box center = (109.84, 127.38, 94.17), size = 24.0 A cube
drd2_inactive: box center = (9.86, 5.87, -9.57), size = 24.0 A cube
cb2_active: box center = (109.71, 109.42, 126.12), size = 24.0 A cube
cb2_inactive: box center = (9.25, -0.23, -55.79), size = 24.0 A cube
adora2a_active: box center = (-3.72, -43.95, 21.20), size = 24.0 A cube
adora2a_inactive: box center = (-9.12, -7.09, 56.20), size = 24.0 A cube
oprm1_active: box center = (185.52, 171.33, 172.71), size = 24.0 A cube
oprm1_inactive: box center = (139.73, 141.81, 185.22), size = 24.0 A cube
ccr5_inactive_allosteric_primary: box center = (149.83, 108.47, 22.33), size = 24.0 A cube
ccr5_inactive_allosteric_comparator: box center = (-3.71, 19.63, 22.33), size = 24.0 A cube

Grid box definitions saved: /content/drive/My Drive/gpcr_benchmark/docking/docking_results/grid_box_definitions.csv


## Module F: Redocking Validation (Hard Gate)

For each unit: redock its own reference ligand back into its own receptor,
up to `MAX_REDOCK_ATTEMPTS` times, varying seed AND exhaustiveness each
retry (not just re-rolling the seed). Stop at the first attempt with
heavy-atom RMSD <= `RMSD_PASS_THRESHOLD` to the crystallographic pose.
**Every attempt is logged, even after the loop exits early** -- the
stopping rule is simple, the disclosure is not, so Methods/SI can report
e.g. "target X needed 4/6 attempts" per unit.

A unit that exhausts all 6 attempts without passing is flagged and
**excluded from candidate docking** -- not silently used with an unvalidated
setup. This is a hard gate, decided before any code was written.

In [14]:
def compute_heavy_atom_rmsd(pose_pdbqt: Path, reference_pdbqt: Path) -> float:
    '''Heavy-atom RMSD between a docked pose and the crystallographic
    reference, via RDKit's substructure/symmetry-aware best-fit RMSD
    (AllChem.GetBestRMS) rather than same-index positional distance.

    Switched from positional RMSD after live cross-verification found it
    systematically overstating failure for oprm1_active: direct positional
    RMSD reported 2.0-4.0 A across every attempt at every box size tested
    (20/24/28/30 A), while RDKit's best-fit RMSD on the IDENTICAL poses
    consistently came back 1.3-1.5 A -- a ~1 A gap present on every single
    attempt, not random noise, so not something a bigger box or more
    exhaustiveness could ever have fixed. Root cause: _correct_bond_orders
    (Module D) reassigns bonds against an RCSB template via
    AssignBondOrdersFromTemplate, which can reindex atom order relative to
    the original extracted file -- positional RMSD then compares atoms
    that no longer correspond 1:1 between pose and reference, even when
    the actual 3D poses agree closely.'''
    def _pdbqt_to_mol(pdbqt_path):
        pdb_path = pdbqt_path.with_suffix('.rmsd_tmp.pdb')
        subprocess.run(['obabel', str(pdbqt_path), '-O', str(pdb_path)], check=True, capture_output=True)
        mol = Chem.MolFromPDBFile(str(pdb_path), removeHs=False)
        return Chem.RemoveHs(mol) if mol is not None else None

    pose_mol = _pdbqt_to_mol(pose_pdbqt)
    ref_mol = _pdbqt_to_mol(reference_pdbqt)
    if pose_mol is None or ref_mol is None:
        return float('inf')
    try:
        return float(AllChem.GetBestRMS(ref_mol, pose_mol))
    except Exception:
        return float('inf')


def run_vina_redock(receptor_pdbqt, ligand_pdbqt, box, seed, exhaustiveness, out_path):
    # --num_modes 1 added defensively: without it Vina defaults to writing
    # up to 9 poses into one output file (MODEL/ENDMDL blocks), and
    # compute_heavy_atom_rmsd's _coords() reads every ATOM/HETATM line with
    # no regard for model boundaries -- inflating the pose atom count ~9x
    # vs the single-pose reference and guaranteeing an atom-count-mismatch
    # (RMSD=inf) on every attempt, independent of the real receptor-
    # contamination bug this was originally mistaken for.
    cmd = ['vina', '--receptor', str(receptor_pdbqt), '--ligand', str(ligand_pdbqt),
           '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
           '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']),
           '--seed', str(seed), '--exhaustiveness', str(exhaustiveness), '--num_modes', '1',
           '--out', str(out_path), '--cpu', str(N_CORES)]
    subprocess.run(cmd, check=True, capture_output=True)
    return out_path


# Reload any already-completed redocking validation from disk FIRST -- this is
# the checkpoint that was missing. Without it, a notebook restart (PBS job
# retry, kernel restart between Module F and Modules G/H) would silently
# redo the full 6-attempt cycle for every unit, including units that already
# passed, wasting real wall-clock time. Same "check disk before recomputing"
# convention already used for receptor prep and ligand prep above.
redocking_validation_path = dockres_dir / shard_name('redocking_validation.csv')
redocking_all_attempts_path = dockres_dir / shard_name('redocking_all_attempts.csv')

if redocking_validation_path.exists():
    redocking_validation_records = pd.read_csv(redocking_validation_path).to_dict('records')
    _already_done_units = {r['unit'] for r in redocking_validation_records}
    print(f'Redocking checkpoint found: {len(_already_done_units)} unit(s) already validated, loading from disk')
else:
    redocking_validation_records = []
    _already_done_units = set()

redocking_all_attempts_records = (
    pd.read_csv(redocking_all_attempts_path).to_dict('records') if redocking_all_attempts_path.exists() else []
)
units_passing_redock = {r['unit'] for r in redocking_validation_records if r['unit_passed']}

for target, state, pdb_id, ref_ligand_code, pocket_type, species, notes in RUN_UNITS:
    unit_key = f'{target}_{state}'
    if unit_key in _already_done_units:
        _status = 'passed' if unit_key in units_passing_redock else 'failed'
        print(f'  \u23ed\ufe0f  {unit_key}: redocking validation checkpoint found, skipping (already {_status})')
        continue
    receptor_pdbqt = protein_dir / f'{unit_key}_{pdb_id}_prepared.pdbqt'
    ref_ligand_pdbqt = ligand_dir / f'{unit_key}_{pdb_id}_{ref_ligand_code}_reference.pdbqt'
    if unit_key not in box_definitions or not receptor_pdbqt.exists() or not ref_ligand_pdbqt.exists():
        print(f'  \u23ed\ufe0f  {unit_key}: missing receptor/ligand/box, skipping redocking validation')
        continue
    box = box_definitions[unit_key]

    attempts = []
    unit_passed = False
    for attempt in range(1, MAX_REDOCK_ATTEMPTS + 1):
        seed, exhaustiveness = redock_seed_and_exhaustiveness(attempt)
        pose_path = dockres_dir / f'redock_{unit_key}_attempt{attempt}.pdbqt'
        try:
            run_vina_redock(receptor_pdbqt, ref_ligand_pdbqt, box, seed, exhaustiveness, pose_path)
            rmsd = compute_heavy_atom_rmsd(pose_path, ref_ligand_pdbqt)
        except Exception as exc:
            rmsd = float('inf')
            log_milestone(unit_key, f'redock attempt {attempt} engine failure: {exc}')

        passed = rmsd <= RMSD_PASS_THRESHOLD
        attempts.append({'unit': unit_key, 'attempt': attempt, 'seed': seed,
                          'exhaustiveness': exhaustiveness, 'rmsd': rmsd, 'passed': passed})
        redocking_all_attempts_records.append(attempts[-1])
        if passed:
            unit_passed = True
            break  # stop at first pass -- explicit decision, not majority-of-N

    best_rmsd = min(a['rmsd'] for a in attempts)
    n_attempts_used = len(attempts)
    redocking_validation_records.append({
        'unit': unit_key, 'target': target, 'state': state, 'pdb_id': pdb_id,
        'n_attempts_used': n_attempts_used, 'best_rmsd': best_rmsd, 'unit_passed': unit_passed,
        'winning_seed': attempts[-1]['seed'] if unit_passed else None,
        'winning_exhaustiveness': attempts[-1]['exhaustiveness'] if unit_passed else None,
    })
    if unit_passed:
        units_passing_redock.add(unit_key)
        log_milestone(unit_key, f'Redocking validation PASSED after {n_attempts_used}/{MAX_REDOCK_ATTEMPTS} attempts, RMSD={best_rmsd:.2f}')
        print(f'  \u2705 {unit_key}: passed after {n_attempts_used}/{MAX_REDOCK_ATTEMPTS} attempts (RMSD={best_rmsd:.2f} A)')
    else:
        log_milestone(unit_key, f'Redocking validation FAILED after {MAX_REDOCK_ATTEMPTS} attempts, best RMSD={best_rmsd:.2f}')
        print(f'  \u274c {unit_key}: FAILED all {MAX_REDOCK_ATTEMPTS} attempts (best RMSD={best_rmsd:.2f} A) '
              f'-- EXCLUDED from candidate docking, flagged for manual receptor-prep review')

    # Incremental per-unit checkpoint -- was previously missing here (unlike
    # Modules B/D), meaning a Colab disconnect mid-run lost every unit's
    # result, including already-passed ones, since nothing hit disk until
    # the WHOLE loop over all 10 units finished. Save after every unit now,
    # same resume-safe convention as receptor/ligand prep.
    pd.DataFrame(redocking_validation_records).to_csv(redocking_validation_path, index=False)
    pd.DataFrame(redocking_all_attempts_records).to_csv(redocking_all_attempts_path, index=False)

redocking_validation_df = pd.DataFrame(redocking_validation_records)
redocking_all_attempts_df = pd.DataFrame(redocking_all_attempts_records)
redocking_validation_df.to_csv(dockres_dir / shard_name('redocking_validation.csv'), index=False)
redocking_all_attempts_df.to_csv(dockres_dir / shard_name('redocking_all_attempts.csv'), index=False)

print(f'\n{len(units_passing_redock)} / {len(RUN_UNITS)} units passed redocking validation')
print(redocking_validation_df[['unit', 'n_attempts_used', 'best_rmsd', 'unit_passed']].to_string(index=False))


Redocking checkpoint found: 10 unit(s) already validated, loading from disk
  ⏭️  drd2_active: redocking validation checkpoint found, skipping (already passed)
  ⏭️  drd2_inactive: redocking validation checkpoint found, skipping (already passed)
  ⏭️  cb2_active: redocking validation checkpoint found, skipping (already passed)
  ⏭️  cb2_inactive: redocking validation checkpoint found, skipping (already passed)
  ⏭️  adora2a_active: redocking validation checkpoint found, skipping (already passed)
  ⏭️  adora2a_inactive: redocking validation checkpoint found, skipping (already passed)
  ⏭️  oprm1_active: redocking validation checkpoint found, skipping (already passed)
  ⏭️  oprm1_inactive: redocking validation checkpoint found, skipping (already passed)
  ⏭️  ccr5_inactive_allosteric_primary: redocking validation checkpoint found, skipping (already passed)
  ⏭️  ccr5_inactive_allosteric_comparator: redocking validation checkpoint found, skipping (already passed)

10 / 10 units passed red

## Module G: AutoDock Vina Docking

Only units that **passed redocking validation** proceed to candidate
docking — a unit that failed the Module F gate is skipped here entirely,
not used with an unvalidated setup. Checkpointed per (unit, candidate)
pair.

In [15]:
vina_results = []
vina_ckpt_path = dockres_dir / shard_name('vina_docking_results.csv')
if vina_ckpt_path.exists():
    _loaded_vina = pd.read_csv(vina_ckpt_path)
    # Checkpoint hygiene: discard stale rows from older candidate sets/CCR5 runs
    # before deciding what is already done. This prevents old rows from leaking
    # into consensus or making a current candidate look already docked.
    _current_pairs = set()
    for _target, _grp in docking_candidates_df.groupby('target'):
        _units = [f'{r[0]}_{r[1]}' for r in RUN_UNITS if r[0] == _target]
        for _unit in _units:
            for _cid in _grp['global_compound_id']:
                _current_pairs.add((_unit, _cid))
    _loaded_vina = _loaded_vina[_loaded_vina.apply(lambda r: (r['unit'], r['global_compound_id']) in _current_pairs, axis=1)]
    vina_results = _loaded_vina.to_dict('records')
    # Only a real 'ok' result counts as done -- a 'failed: ...' row must be
    # retried on rerun, not treated as permanently skipped (same class of
    # bug found live in the GNINA checkpoint after a 903-compound outage).
    vina_results = [r for r in vina_results if r.get('status') == 'ok']
    _done_pairs = {(r['unit'], r['global_compound_id']) for r in vina_results}
else:
    vina_results = []
    _done_pairs = set()

for target, state, pdb_id, ref_ligand_code, pocket_type, species, notes in RUN_UNITS:
    unit_key = f'{target}_{state}'
    if unit_key not in units_passing_redock:
        print(f'  \u23ed\ufe0f  {unit_key}: did not pass redocking validation, skipping Vina docking entirely')
        continue
    receptor_pdbqt = protein_dir / f'{unit_key}_{pdb_id}_prepared.pdbqt'
    box = box_definitions[unit_key]

    target_candidates = docking_candidates_df[docking_candidates_df.target == target]
    for _, cand in target_candidates.iterrows():
        cid = cand['global_compound_id']
        if (unit_key, cid) in _done_pairs:
            continue
        ligand_pdbqt = ligand_dir / f'candidate_{cid}.pdbqt'
        if not ligand_pdbqt.exists():
            continue
        bad_types = invalid_pdbqt_atom_types(ligand_pdbqt)
        if bad_types:
            print(f'  ⚠️  {ligand_pdbqt.name}: invalid PDBQT atom types {bad_types[:5]}, skipping docking until ligand prep is rerun')
            continue
        out_path = dockres_dir / f'vina_{unit_key}_{cid}.pdbqt'
        try:
            run_vina_redock(receptor_pdbqt, ligand_pdbqt, box, BASE_SEED, VINA_EXHAUSTIVENESS, out_path)
            # Vina writes per-pose affinity in REMARK lines of the output PDBQT;
            # parse the best (first) pose's affinity.
            best_affinity = None
            with open(out_path) as f:
                for line in f:
                    if line.startswith('REMARK VINA RESULT'):
                        best_affinity = float(line.split()[3])
                        break
            vina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                  'global_compound_id': cid, 'vina_affinity': best_affinity,
                                  'status': 'ok'})
        except Exception as exc:
            vina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                  'global_compound_id': cid, 'vina_affinity': None,
                                  'status': f'failed: {exc}'})

    # drop_duplicates guards against exactly the disconnect/resume
    # scenario that produced 34 duplicate rows in a live run -- keeps
    # this checkpoint self-healing on every save, not just fixed once
    # downstream in Module I.
    pd.DataFrame(vina_results).drop_duplicates(
        subset=['unit', 'target', 'state', 'global_compound_id'], keep='last'
    ).to_csv(vina_ckpt_path, index=False)  # checkpoint after each unit
    print(f'{unit_key}: Vina docking complete for {len(target_candidates)} candidates')

vina_df = pd.DataFrame(vina_results)
print(f'\nTotal Vina results: {len(vina_df)}')


drd2_active: Vina docking complete for 20 candidates
drd2_inactive: Vina docking complete for 20 candidates
cb2_active: Vina docking complete for 2 candidates
cb2_inactive: Vina docking complete for 2 candidates
adora2a_active: Vina docking complete for 2 candidates
adora2a_inactive: Vina docking complete for 2 candidates
oprm1_active: Vina docking complete for 20 candidates
oprm1_inactive: Vina docking complete for 20 candidates
ccr5_inactive_allosteric_primary: Vina docking complete for 0 candidates
ccr5_inactive_allosteric_comparator: Vina docking complete for 0 candidates

Total Vina results: 88


## Module H: GNINA Docking

Same box, same candidate set, same units (only those passing Module F).
GNINA's CNN-based rescoring is a genuinely different scoring philosophy
from Vina's empirical function — that difference is what makes agreement
between the two meaningful.

In [16]:
_gnina_health_checked = False

def ensure_gnina_available():
    '''Self-heals gnina availability. Rewritten after live evidence showed
    the first version was checking the wrong thing: it only verified the
    binary FILE exists, but the real failure (confirmed via error-message
    shape) is that the file exists and is executable, yet fails to launch --
    subprocess.run() with a list command (not shell=True) raises a Python
    FileNotFoundError directly if the executable is truly missing from
    PATH; seeing a CalledProcessError with "exit status 127" instead means
    the process launched and something INSIDE its own exec path failed --
    the signature of a missing shared library at load time (the same class
    of problem hit on the cluster: libcudnn.so.9/libcudart.so.12 missing),
    not a missing file. Re-downloading the identical binary cannot fix a
    missing shared library, so this now (a) does a REAL functional check
    (gnina --version actually runs, not just os.path.exists), and (b)
    re-fetches only as a cheap first attempt in case the file genuinely
    was removed -- run_gnina()'s --no_gpu retry (below) is the real fix
    for the shared-library case, since CPU-only mode may not dlopen cuDNN
    at all. Checked once per session (module-level flag), not on every
    call -- a real subprocess health check is not free like a file-exists
    check was.'''
    global _gnina_health_checked
    if _gnina_health_checked:
        return
    result = subprocess.run(['gnina', '--version'], capture_output=True)
    if result.returncode == 0:
        _gnina_health_checked = True
        return
    stderr_text = result.stderr.decode('utf-8', errors='replace').strip()
    print(f'  [self-heal] gnina --version failed (exit {result.returncode}): {stderr_text[:500]}')
    print('  [self-heal] re-fetching binary as a first attempt...')
    gnina_path = '/content/gnina' if not HPC_MODE else './gnina'
    with urllib.request.urlopen('https://api.github.com/repos/gnina/gnina/releases/latest') as resp:
        release = json.load(resp)
    matching_assets = [a for a in release['assets'] if a['name'].startswith('gnina')]
    if not matching_assets:
        raise RuntimeError(f"No asset starting with 'gnina' found in release {release['tag_name']}")
    asset = matching_assets[0]
    urllib.request.urlretrieve(asset['browser_download_url'], gnina_path)
    st = os.stat(gnina_path)
    os.chmod(gnina_path, st.st_mode | stat.S_IEXEC)
    subprocess.run(['ln', '-sf', os.path.abspath(gnina_path), '/usr/local/bin/gnina'], check=True)
    result2 = subprocess.run(['gnina', '--version'], capture_output=True)
    if result2.returncode == 0:
        print('  [self-heal] re-fetch fixed it.')
    else:
        stderr_text2 = result2.stderr.decode('utf-8', errors='replace').strip()
        print(f'  [self-heal] re-fetch did NOT fix it (exit {result2.returncode}): {stderr_text2[:500]}')
        print('  [self-heal] run_gnina will fall back to --no_gpu per call instead.')
    _gnina_health_checked = True


def _run_gnina_cmd(cmd):
    '''Runs a gnina command, and on failure DECODES AND PRINTS stderr before
    raising -- capture_output=True was silently swallowing the real error
    text this whole time. str(CalledProcessError) only shows the command
    and return code, never stderr, unless accessed explicitly -- this is
    why every failure so far only ever showed "exit status 127" with no
    indication of WHY. Re-raises with stderr appended to the message so it
    also reaches the persisted CSV status column, not just stdout.'''
    result = subprocess.run(cmd, capture_output=True)
    if result.returncode != 0:
        stderr_text = result.stderr.decode('utf-8', errors='replace').strip()
        stdout_text = result.stdout.decode('utf-8', errors='replace').strip()
        print(f'  [gnina real error] exit {result.returncode}')
        print(f'  [gnina stderr] {stderr_text[:500]}')
        if stdout_text:
            print(f'  [gnina stdout] {stdout_text[:300]}')
        raise subprocess.CalledProcessError(
            result.returncode, cmd, output=result.stdout,
            stderr=(stderr_text[:500] or '(empty stderr)').encode()
        )


def run_gnina(receptor_pdbqt, ligand_pdbqt, box, out_path, seed=BASE_SEED, _retry_no_gpu=True):
    ensure_gnina_available()
    cmd = ['gnina', '-r', str(receptor_pdbqt), '-l', str(ligand_pdbqt),
           '--center_x', str(box['center_x']), '--center_y', str(box['center_y']), '--center_z', str(box['center_z']),
           '--size_x', str(box['size_x']), '--size_y', str(box['size_y']), '--size_z', str(box['size_z']),
           '--seed', str(seed), '--num_modes', '1', '-o', str(out_path), '--cpu', str(N_CORES)]
    try:
        _run_gnina_cmd(cmd)
        return out_path
    except subprocess.CalledProcessError as exc:
        first_stderr = exc.stderr.decode('utf-8', errors='replace') if exc.stderr else '(empty stderr)'
        if not _retry_no_gpu:
            # str(CalledProcessError) does NOT include .stderr by default --
            # confirmed live (2026-08) that the caller's f'failed: {exc}'
            # formatting was silently dropping the real error text into just
            # "exit status N", the entire reason this was hard to diagnose.
            # Re-raise as RuntimeError so the real message actually reaches
            # the persisted CSV status column, not just the live print above.
            raise RuntimeError(f'gnina failed (exit {exc.returncode}): {first_stderr}') from exc
        # Automatic CPU fallback: a missing GPU shared library (cuDNN) only
        # affects GPU-mode CNN scoring -- --no_gpu may avoid loading it
        # entirely and succeed where the default (GPU-if-available) call
        # just failed. Retried once, not looped, so a genuinely bad
        # ligand/receptor pair still fails cleanly rather than retrying
        # forever.
        print(f'  [fallback] gnina failed (exit {exc.returncode}), retrying once with --no_gpu...')
        cmd_cpu = cmd + ['--no_gpu']
        try:
            _run_gnina_cmd(cmd_cpu)
            return out_path
        except subprocess.CalledProcessError as exc2:
            second_stderr = exc2.stderr.decode('utf-8', errors='replace') if exc2.stderr else '(empty stderr)'
            raise RuntimeError(
                f'gnina failed on both default (exit {exc.returncode}: {first_stderr}) '
                f'and --no_gpu (exit {exc2.returncode}: {second_stderr})'
            ) from exc2


gnina_results = []
gnina_ckpt_path = dockres_dir / shard_name('gnina_docking_results.csv')
if gnina_ckpt_path.exists():
    _loaded_gnina = pd.read_csv(gnina_ckpt_path)
    # Same stale-checkpoint hygiene as Vina: keep only pairs belonging to the
    # current docking candidate set and current receptor units.
    _current_pairs = set()
    for _target, _grp in docking_candidates_df.groupby('target'):
        _units = [f'{r[0]}_{r[1]}' for r in RUN_UNITS if r[0] == _target]
        for _unit in _units:
            for _cid in _grp['global_compound_id']:
                _current_pairs.add((_unit, _cid))
    _loaded_gnina = _loaded_gnina[_loaded_gnina.apply(lambda r: (r['unit'], r['global_compound_id']) in _current_pairs, axis=1)]
    gnina_results = _loaded_gnina.to_dict('records')
    # Only a real 'ok' result counts as done -- a 'failed: ...' row must be
    # retried on rerun, not treated as permanently skipped. Bug found live:
    # without this, the checkpoint would silently make the 903-compound
    # GNINA outage from the enrichment run permanent, since every failed
    # pair already has a row and would never be re-attempted.
    gnina_results = [r for r in gnina_results if r.get('status') == 'ok']
    _done_pairs = {(r['unit'], r['global_compound_id']) for r in gnina_results}
else:
    gnina_results = []
    _done_pairs = set()

for target, state, pdb_id, ref_ligand_code, pocket_type, species, notes in RUN_UNITS:
    unit_key = f'{target}_{state}'
    if unit_key not in units_passing_redock:
        print(f'  \u23ed\ufe0f  {unit_key}: did not pass redocking validation, skipping GNINA docking entirely')
        continue
    receptor_pdbqt = protein_dir / f'{unit_key}_{pdb_id}_prepared.pdbqt'
    box = box_definitions[unit_key]

    target_candidates = docking_candidates_df[docking_candidates_df.target == target]
    for _, cand in target_candidates.iterrows():
        cid = cand['global_compound_id']
        if (unit_key, cid) in _done_pairs:
            continue
        ligand_pdbqt = ligand_dir / f'candidate_{cid}.pdbqt'
        if not ligand_pdbqt.exists():
            continue
        bad_types = invalid_pdbqt_atom_types(ligand_pdbqt)
        if bad_types:
            print(f'  ⚠️  {ligand_pdbqt.name}: invalid PDBQT atom types {bad_types[:5]}, skipping docking until ligand prep is rerun')
            continue
        out_path = dockres_dir / f'gnina_{unit_key}_{cid}.sdf'
        try:
            run_gnina(receptor_pdbqt, ligand_pdbqt, box, out_path)
            cnn_score = None
            # sanitize=False: confirmed live that GNINA's own OpenBabel-
            # based SDF writer drops formal-charge annotations on
            # protonated basic amines (e.g. a piperidine N+ ends up written
            # as a neutral N with 4 bonds), which RDKit's default sanitizer
            # correctly rejects as invalid valence -- even though the actual
            # 3D pose and the CNNscore property tag are both fine. Reading
            # CNNscore is pure property/metadata access, not a chemistry
            # operation, so it needs no valid chemical graph at all.
            for mol in Chem.SDMolSupplier(str(out_path), sanitize=False):
                if mol is not None and mol.HasProp('CNNscore'):
                    cnn_score = float(mol.GetProp('CNNscore'))
                    break
            gnina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                   'global_compound_id': cid, 'gnina_cnn_score': cnn_score,
                                   'status': 'ok'})
        except Exception as exc:
            gnina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                   'global_compound_id': cid, 'gnina_cnn_score': None,
                                   'status': f'failed: {exc}'})

    # Same self-healing dedup as Module G's checkpoint save.
    pd.DataFrame(gnina_results).drop_duplicates(
        subset=['unit', 'target', 'state', 'global_compound_id'], keep='last'
    ).to_csv(gnina_ckpt_path, index=False)
    print(f'{unit_key}: GNINA docking complete for {len(target_candidates)} candidates')

gnina_df = pd.DataFrame(gnina_results)
print(f'\nTotal GNINA results: {len(gnina_df)}')


drd2_active: GNINA docking complete for 20 candidates
drd2_inactive: GNINA docking complete for 20 candidates
cb2_active: GNINA docking complete for 2 candidates
cb2_inactive: GNINA docking complete for 2 candidates
adora2a_active: GNINA docking complete for 2 candidates
adora2a_inactive: GNINA docking complete for 2 candidates
oprm1_active: GNINA docking complete for 20 candidates
oprm1_inactive: GNINA docking complete for 20 candidates
ccr5_inactive_allosteric_primary: GNINA docking complete for 0 candidates
ccr5_inactive_allosteric_comparator: GNINA docking complete for 0 candidates

Total GNINA results: 88


## Module I: GNINA-Vina Consensus

A candidate "passes" a unit if **both** engines rank it within the top
`CONSENSUS_TOP_PERCENT` of that unit's docked set — defined in advance in
the config cell, not chosen after seeing the score distribution.

In [17]:
# Real duplicate rows confirmed in both checkpoint CSVs (~17 (unit,
# compound) pairs recorded more than once -- likely a Colab disconnect/
# resume artifact) -- dedupe before merging rather than after, since an
# outer merge on duplicate keys cross-multiplies rows, which was silently
# turning a handful of real GNINA failures into ~92 NaN rows post-merge
# and understating consensus-eligible pairs (116 instead of the true 150).
# Prefers a row with a real (non-null) score over a duplicate that
# happened to fail, rather than blindly keeping file order.
_key_cols = ['unit', 'target', 'state', 'global_compound_id']
_current_pairs_df = []
for _target, _grp in docking_candidates_df.groupby('target'):
    for _target2, _state, _pdb_id, _lig, _pocket, _species, _notes in RUN_UNITS:
        if _target2 != _target:
            continue
        _unit = f'{_target2}_{_state}'
        for _cid in _grp['global_compound_id']:
            _current_pairs_df.append({'unit': _unit, 'target': _target, 'state': _state, 'global_compound_id': _cid})
_current_pairs_df = pd.DataFrame(_current_pairs_df)

vina_dedup = vina_df.sort_values('vina_affinity', key=lambda s: s.isna()).drop_duplicates(subset=_key_cols, keep='first')
gnina_dedup = gnina_df.sort_values('gnina_cnn_score', key=lambda s: s.isna()).drop_duplicates(subset=_key_cols, keep='first')
vina_dedup = _current_pairs_df.merge(vina_dedup, on=_key_cols, how='inner')
gnina_dedup = _current_pairs_df.merge(gnina_dedup, on=_key_cols, how='inner')
print(f'Deduplicated/current-filtered: vina {len(vina_df)} -> {len(vina_dedup)}, gnina {len(gnina_df)} -> {len(gnina_dedup)}')

merged_scores = _current_pairs_df.merge(vina_dedup, on=_key_cols, how='left').merge(gnina_dedup, on=_key_cols, suffixes=('_vina', '_gnina'), how='left')

consensus_records = []
for unit_key, grp in merged_scores.groupby('unit'):
    grp = grp.dropna(subset=['vina_affinity', 'gnina_cnn_score'])
    if len(grp) == 0:
        continue
    # Vina affinity: more negative = better, so rank ascending.
    # GNINA CNN score: higher = better, so rank descending.
    n_top = max(1, int(np.ceil(len(grp) * CONSENSUS_TOP_PERCENT / 100)))
    vina_top = set(grp.nsmallest(n_top, 'vina_affinity')['global_compound_id'])
    gnina_top = set(grp.nlargest(n_top, 'gnina_cnn_score')['global_compound_id'])
    for _, row in grp.iterrows():
        consensus_records.append({
            'unit': unit_key, 'target': row['target'], 'state': row['state'],
            'global_compound_id': row['global_compound_id'],
            'vina_affinity': row['vina_affinity'], 'gnina_cnn_score': row['gnina_cnn_score'],
            'in_vina_top_pct': row['global_compound_id'] in vina_top,
            'in_gnina_top_pct': row['global_compound_id'] in gnina_top,
            'consensus_pass': (row['global_compound_id'] in vina_top) and (row['global_compound_id'] in gnina_top),
        })

docking_consensus_df = pd.DataFrame(consensus_records)
docking_consensus_df.to_csv(dockres_dir / shard_name('docking_consensus.csv'), index=False)
n_pass = int(docking_consensus_df['consensus_pass'].sum()) if len(docking_consensus_df) else 0
print(f'Consensus passes: {n_pass} / {len(docking_consensus_df)} (unit, candidate) pairs')


Deduplicated/current-filtered: vina 88 -> 88, gnina 88 -> 88
Consensus passes: 9 / 88 (unit, candidate) pairs


## Module J: Target-Aware Interaction Plausibility Checks

Lightweight, coded checks -- not a full Discovery-Studio-style PLIF.
The aim is to keep the docking interpretation honest: a consensus-scoring pose
should also make chemically plausible contacts for that receptor class. These
checks are deliberately simple, transparent, and reproducible from the prepared
receptor PDBQT plus the docked Vina pose.

Target-aware rules used here:
- **DRD2 / OPRM1**: require a ligand cationic/basic nitrogen near an acidic
  receptor side chain (Asp/Glu) as a proxy for the conserved aminergic/opioid
  salt-bridge anchor. This does not assume an author residue number; it reports
  the nearest acidic residues observed in the prepared receptor.
- **ADORA2A**: require at least one close polar protein-ligand contact, because
  the orthosteric adenosine pocket is not interpretable as a purely hydrophobic
  score-only site.
- **CB2 / CCR5 allosteric**: require substantial hydrophobic packing plus real
  protein contact, matching the lipophilic cannabinoid pocket and maraviroc-like
  CCR5 allosteric pocket.
- **All targets**: require nonzero close protein contact; a pose floating in the
  grid box is rejected regardless of score.

These are plausibility filters, not experimental proof of binding.


In [18]:
# Target-aware interaction plausibility checks.
# These checks are intentionally transparent and conservative. They do not try
# to replace a full interaction-fingerprint package; they make the minimum
# receptor-class pharmacology assumptions explicit and auditable.
GENERIC_CONTACT_CUTOFF = 4.5
POLAR_CONTACT_CUTOFF = 3.5
IONIC_CONTACT_CUTOFF = 4.0
HYDROPHOBIC_CONTACT_CUTOFF = 4.5
MIN_HYDROPHOBIC_CONTACTS_LIPIDIC_POCKET = 10

ACIDIC_RESNAMES = {'ASP', 'GLU'}
ACIDIC_SIDECHAIN_ATOMS = {'OD1', 'OD2', 'OE1', 'OE2'}
POLAR_ELEMENTS = {'N', 'O', 'S'}
HYDROPHOBIC_ELEMENTS = {'C', 'S'}
POSITIVE_N_TYPES = {'N', 'NA'}  # PDBQT atom types commonly used for ligand nitrogens.

TARGET_PLIF_RULES = {
    'drd2': {
        'rule': 'aminergic acidic-anchor proxy',
        'requires_ionic_acid_contact': True,
        'requires_polar_contact': False,
        'min_hydrophobic_contacts': 0,
        'note': 'Cationic/basic ligand nitrogen near receptor Asp/Glu side chain; reports observed residues instead of assuming Asp3.32 author numbering.',
    },
    'oprm1': {
        'rule': 'opioid acidic-anchor proxy',
        'requires_ionic_acid_contact': True,
        'requires_polar_contact': False,
        'min_hydrophobic_contacts': 0,
        'note': 'Cationic/basic ligand nitrogen near receptor Asp/Glu side chain; proxy for conserved opioid receptor salt-bridge anchoring.',
    },
    'adora2a': {
        'rule': 'adenosine-pocket polar contact',
        'requires_ionic_acid_contact': False,
        'requires_polar_contact': True,
        'min_hydrophobic_contacts': 0,
        'note': 'At least one close polar contact in the adenosine orthosteric pocket.',
    },
    'cb2': {
        'rule': 'lipophilic cannabinoid-pocket packing',
        'requires_ionic_acid_contact': False,
        'requires_polar_contact': False,
        'min_hydrophobic_contacts': MIN_HYDROPHOBIC_CONTACTS_LIPIDIC_POCKET,
        'note': 'Requires substantial hydrophobic packing; CB2 docking scores alone are treated cautiously.',
    },
    'ccr5': {
        'rule': 'maraviroc-like allosteric pocket packing',
        'requires_ionic_acid_contact': False,
        'requires_polar_contact': False,
        'min_hydrophobic_contacts': MIN_HYDROPHOBIC_CONTACTS_LIPIDIC_POCKET,
        'note': 'Requires substantial hydrophobic packing in the CCR5 allosteric pocket.',
    },
}


def _pdbqt_element(line):
    elem = line[76:78].strip().upper()
    if elem:
        return elem
    atom_type = line[77:].strip().split()[0].upper() if len(line) > 77 and line[77:].strip() else ''
    if atom_type:
        return ''.join(ch for ch in atom_type if ch.isalpha())[:2].upper()
    atom_name = line[12:16].strip().upper()
    return ''.join(ch for ch in atom_name if ch.isalpha())[:2].upper()


def _parse_pdbqt_atoms(path: Path, receptor=False):
    atoms = []
    with open(path) as f:
        for line in f:
            if not line.startswith(('ATOM', 'HETATM')):
                continue
            elem = _pdbqt_element(line)
            if elem == 'H':
                continue
            atom = {
                'x': float(line[30:38]), 'y': float(line[38:46]), 'z': float(line[46:54]),
                'element': elem, 'atom_name': line[12:16].strip(),
                'resname': line[17:20].strip(), 'chain': line[21].strip(),
                'resi': line[22:26].strip(),
            }
            atoms.append(atom)
    return atoms


def _coords(atoms):
    if not atoms:
        return np.empty((0, 3), dtype=float)
    return np.array([[a['x'], a['y'], a['z']] for a in atoms], dtype=float)


def _distance_matrix(a_atoms, b_atoms):
    ac = _coords(a_atoms); bc = _coords(b_atoms)
    if len(ac) == 0 or len(bc) == 0:
        return np.empty((len(a_atoms), len(b_atoms)))
    return np.linalg.norm(ac[:, None, :] - bc[None, :, :], axis=2)


def _residue_label(atom):
    chain = atom.get('chain') or '?'
    return f"{atom.get('resname', '?')}:{chain}:{atom.get('resi', '?')}"


def _summarize_residues(atoms, max_items=5):
    labels = []
    for atom in atoms:
        label = _residue_label(atom)
        if label not in labels:
            labels.append(label)
        if len(labels) >= max_items:
            break
    return ';'.join(labels)


def interaction_plausibility(pose_path: Path, receptor_pdbqt: Path, target: str):
    ligand_atoms = _parse_pdbqt_atoms(pose_path)
    receptor_atoms = _parse_pdbqt_atoms(receptor_pdbqt, receptor=True)
    dmat = _distance_matrix(ligand_atoms, receptor_atoms)

    if len(ligand_atoms) == 0 or len(receptor_atoms) == 0:
        return {
            'n_ligand_heavy_atoms': len(ligand_atoms), 'n_receptor_heavy_atoms': len(receptor_atoms),
            'n_protein_contacts': 0, 'n_polar_contacts': 0, 'n_hydrophobic_contacts': 0,
            'n_ionic_acid_contacts': 0, 'nearest_acidic_residue': None,
            'nearest_acidic_distance': np.nan, 'target_plif_rule': TARGET_PLIF_RULES.get(target, {}).get('rule', 'generic contact'),
            'target_plif_note': TARGET_PLIF_RULES.get(target, {}).get('note', 'generic contact only'),
            'passes_generic_contact_check': False, 'passes_target_aware_plif_check': False,
            'target_plif_failure_reason': 'missing ligand or receptor atoms',
        }

    contact_mask = dmat <= GENERIC_CONTACT_CUTOFF
    n_contacts = int(contact_mask.sum())

    lig_polar_idx = [i for i, a in enumerate(ligand_atoms) if a['element'][:1] in POLAR_ELEMENTS]
    rec_polar_idx = [i for i, a in enumerate(receptor_atoms) if a['element'][:1] in POLAR_ELEMENTS]
    n_polar = 0
    if lig_polar_idx and rec_polar_idx:
        n_polar = int((dmat[np.ix_(lig_polar_idx, rec_polar_idx)] <= POLAR_CONTACT_CUTOFF).sum())

    lig_hydro_idx = [i for i, a in enumerate(ligand_atoms) if a['element'][:1] in HYDROPHOBIC_ELEMENTS]
    rec_hydro_idx = [i for i, a in enumerate(receptor_atoms) if a['element'][:1] in HYDROPHOBIC_ELEMENTS]
    n_hydro = 0
    if lig_hydro_idx and rec_hydro_idx:
        n_hydro = int((dmat[np.ix_(lig_hydro_idx, rec_hydro_idx)] <= HYDROPHOBIC_CONTACT_CUTOFF).sum())

    lig_pos_n_idx = [i for i, a in enumerate(ligand_atoms) if a['element'] in POSITIVE_N_TYPES or a['atom_name'].upper().startswith('N')]
    rec_acid_idx = [i for i, a in enumerate(receptor_atoms)
                    if a['resname'] in ACIDIC_RESNAMES and a['atom_name'].upper() in ACIDIC_SIDECHAIN_ATOMS]
    n_ionic = 0
    nearest_acid_label = None
    nearest_acid_dist = np.nan
    acid_contacts = []
    if lig_pos_n_idx and rec_acid_idx:
        acid_d = dmat[np.ix_(lig_pos_n_idx, rec_acid_idx)]
        n_ionic = int((acid_d <= IONIC_CONTACT_CUTOFF).sum())
        min_pos = np.unravel_index(np.argmin(acid_d), acid_d.shape)
        nearest_acid_dist = float(acid_d[min_pos])
        nearest_acid_atom = receptor_atoms[rec_acid_idx[min_pos[1]]]
        nearest_acid_label = _residue_label(nearest_acid_atom)
        contact_rec_positions = sorted(set(rec_acid_idx[j] for _, j in zip(*np.where(acid_d <= IONIC_CONTACT_CUTOFF))))
        acid_contacts = [receptor_atoms[j] for j in contact_rec_positions]

    rule = TARGET_PLIF_RULES.get(target, {'rule': 'generic contact', 'note': 'generic contact only',
                                          'requires_ionic_acid_contact': False, 'requires_polar_contact': False,
                                          'min_hydrophobic_contacts': 0})
    failures = []
    if n_contacts == 0:
        failures.append('no close protein contact')
    if rule.get('requires_ionic_acid_contact') and n_ionic == 0:
        failures.append('no ligand-N to receptor Asp/Glu side-chain contact')
    if rule.get('requires_polar_contact') and n_polar == 0:
        failures.append('no close polar contact')
    min_hydro = int(rule.get('min_hydrophobic_contacts', 0))
    if n_hydro < min_hydro:
        failures.append(f'hydrophobic contacts below target threshold ({n_hydro} < {min_hydro})')

    return {
        'n_ligand_heavy_atoms': len(ligand_atoms), 'n_receptor_heavy_atoms': len(receptor_atoms),
        'n_protein_contacts': n_contacts, 'n_polar_contacts': n_polar,
        'n_hydrophobic_contacts': n_hydro, 'n_ionic_acid_contacts': n_ionic,
        'ionic_contact_residues': _summarize_residues(acid_contacts),
        'nearest_acidic_residue': nearest_acid_label,
        'nearest_acidic_distance': nearest_acid_dist,
        'target_plif_rule': rule.get('rule'), 'target_plif_note': rule.get('note'),
        'passes_generic_contact_check': n_contacts > 0,
        'passes_target_aware_plif_check': len(failures) == 0,
        'target_plif_failure_reason': '; '.join(failures) if failures else 'passes target-aware plausibility rule',
    }


def count_protein_contacts(pose_path: Path, receptor_pdbqt: Path, cutoff=GENERIC_CONTACT_CUTOFF) -> int:
    # Backward-compatible helper used by the rediscovery module.
    ligand_atoms = _parse_pdbqt_atoms(pose_path)
    receptor_atoms = _parse_pdbqt_atoms(receptor_pdbqt, receptor=True)
    dmat = _distance_matrix(ligand_atoms, receptor_atoms)
    return int((dmat <= cutoff).sum()) if dmat.size else 0


plif_records = []
for _, row in docking_consensus_df[docking_consensus_df['consensus_pass']].iterrows():
    unit_key, cid, target = row['unit'], row['global_compound_id'], row['target']
    pose_path = dockres_dir / f'vina_{unit_key}_{cid}.pdbqt'
    pdb_id = design_df[(design_df.target == target) & (design_df.state == row['state'])]['pdb_id'].iloc[0]
    receptor_pdbqt = protein_dir / f'{unit_key}_{pdb_id}_prepared.pdbqt'
    if not (pose_path.exists() and receptor_pdbqt.exists()):
        continue
    metrics = interaction_plausibility(pose_path, receptor_pdbqt, target)
    plif_records.append({
        'unit': unit_key, 'target': target, 'global_compound_id': cid,
        **metrics,
    })

plif_df = pd.DataFrame(plif_records)
plif_df.to_csv(dockres_dir / shard_name('target_aware_plif_plausibility_check.csv'), index=False)
# Keep the old filename too, so downstream/manifest paths remain stable.
plif_df.to_csv(dockres_dir / shard_name('plif_plausibility_check.csv'), index=False)
n_generic = int(plif_df['passes_generic_contact_check'].sum()) if len(plif_df) else 0
n_target = int(plif_df['passes_target_aware_plif_check'].sum()) if len(plif_df) else 0
print(f'Target-aware PLIF plausibility: {n_target} / {len(plif_df)} consensus-passing poses pass target-specific rules')
print(f'Generic contact sanity check: {n_generic} / {len(plif_df)} consensus-passing poses have real protein contact')
if len(plif_df):
    print(plif_df[['unit', 'global_compound_id', 'target_plif_rule', 'n_protein_contacts',
                   'n_polar_contacts', 'n_hydrophobic_contacts', 'n_ionic_acid_contacts',
                   'nearest_acidic_residue', 'nearest_acidic_distance',
                   'passes_target_aware_plif_check', 'target_plif_failure_reason']].to_string(index=False))



Target-aware PLIF plausibility: 5 / 9 consensus-passing poses pass target-specific rules
Generic contact sanity check: 9 / 9 consensus-passing poses have real protein contact
            unit global_compound_id                      target_plif_rule  n_protein_contacts  n_polar_contacts  n_hydrophobic_contacts  n_ionic_acid_contacts nearest_acidic_residue  nearest_acidic_distance  passes_target_aware_plif_check                         target_plif_failure_reason
adora2a_inactive  CMPD_57834cbc78ba        adenosine-pocket polar contact                 195                 8                       0                      0               GLU:A:14                 4.388065                            True              passes target-aware plausibility rule
      cb2_active  CMPD_f67136ec3d33 lipophilic cannabinoid-pocket packing                 170                 1                      43                      0                   None                      NaN                            True       

## Module K: Structural Alerts and Final Consolidated Output

Reuses the same PAINS/Brenk/Lipinski conventions from notebook 2's curation
and notebook 4's screening — not reinvented per-notebook.

In [19]:
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams

params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
alert_catalog = FilterCatalog.FilterCatalog(params)

# Muegge (Muegge et al. 2001) bioavailability filter -- RDKit's FilterCatalog
# has no built-in MUEGGE entry, implemented directly. Ported from the
# predecessor project's 06_docking.ipynb, which itself carries a fix for an
# earlier draft that only implemented 7 of the 9 criteria (missing the >4
# carbons and >1 heteroatom checks) -- reusing the already-corrected version,
# not the buggy earlier one. Descriptors/Crippen/Lipinski are already
# imported module-wide (cell 3), same convention notebook 3 uses for its own
# descriptor computation -- no new imports needed.
def passes_muegge(mol):
    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    tpsa = Descriptors.TPSA(mol)
    rings = Descriptors.RingCount(mol)
    rot_bonds = Descriptors.NumRotatableBonds(mol)
    hba = Lipinski.NumHAcceptors(mol)
    hbd = Lipinski.NumHDonors(mol)
    num_carbons = sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() == 'C')
    num_heteroatoms = sum(1 for atom in mol.GetAtoms() if atom.GetSymbol() not in ('C', 'H'))
    return (
        200 <= mw <= 600 and
        -2 <= logp <= 5 and
        tpsa <= 150 and
        rings <= 7 and
        num_carbons > 4 and
        num_heteroatoms > 1 and
        rot_bonds <= 15 and
        hba <= 10 and
        hbd <= 5
    )


def passes_lipinski(mol):
    return (
        Descriptors.MolWt(mol) <= 500 and Crippen.MolLogP(mol) <= 5 and
        Lipinski.NumHDonors(mol) <= 5 and Lipinski.NumHAcceptors(mol) <= 10
    )


receptor_status_lookup = (receptor_prep_df.set_index('unit').to_dict('index')
                          if len(receptor_prep_df) and 'unit' in receptor_prep_df.columns else {})

final_records = []
for _, row in docking_consensus_df.iterrows():
    cand_row = docking_candidates_df[
        (docking_candidates_df.target == row['target'])
        & (docking_candidates_df.global_compound_id == row['global_compound_id'])
    ]
    if len(cand_row) == 0:
        continue
    smiles = cand_row.iloc[0]['clean_smiles']
    mol = Chem.MolFromSmiles(smiles)
    has_alert = alert_catalog.HasMatch(mol) if mol is not None else None
    lipinski_pass = passes_lipinski(mol) if mol is not None else None
    muegge_pass = passes_muegge(mol) if mol is not None else None

    plif_row = plif_df[(plif_df.unit == row['unit']) & (plif_df.global_compound_id == row['global_compound_id'])]
    final_records.append({
        **row.to_dict(),
        'generic_name': cand_row.iloc[0].get('generic_name'),
        'drugbank_id': cand_row.iloc[0].get('drugbank_id'),
        'priority_score_from_notebook4': cand_row.iloc[0].get('priority_score'),
        'is_unanimous_consensus_notebook4': cand_row.iloc[0].get('is_unanimous_consensus'),
        'receptor_prep_provisional': receptor_status_lookup.get(row['unit'], {}).get('receptor_prep_provisional'),
        'fusion_boundary_confirmed': receptor_status_lookup.get(row['unit'], {}).get('fusion_boundary_confirmed'),
        'fusion_excision_range': receptor_status_lookup.get(row['unit'], {}).get('fusion_excision_range'),
        'has_pains_or_brenk_alert': has_alert,
        'lipinski_pass': lipinski_pass,
        'muegge_pass': muegge_pass,
        'n_protein_contacts': plif_row['n_protein_contacts'].iloc[0] if len(plif_row) else None,
        'n_polar_contacts': plif_row['n_polar_contacts'].iloc[0] if len(plif_row) else None,
        'n_hydrophobic_contacts': plif_row['n_hydrophobic_contacts'].iloc[0] if len(plif_row) else None,
        'n_ionic_acid_contacts': plif_row['n_ionic_acid_contacts'].iloc[0] if len(plif_row) else None,
        'nearest_acidic_residue': plif_row['nearest_acidic_residue'].iloc[0] if len(plif_row) else None,
        'nearest_acidic_distance': plif_row['nearest_acidic_distance'].iloc[0] if len(plif_row) else None,
        'target_plif_rule': plif_row['target_plif_rule'].iloc[0] if len(plif_row) else None,
        'target_plif_failure_reason': plif_row['target_plif_failure_reason'].iloc[0] if len(plif_row) else None,
        'passes_generic_contact_check': plif_row['passes_generic_contact_check'].iloc[0] if len(plif_row) else None,
        'passes_plif_check': plif_row['passes_target_aware_plif_check'].iloc[0] if len(plif_row) else None,
    })

docking_results_final_df = pd.DataFrame(final_records)

# Final-prioritization persistence: consensus pass + no PAINS/Brenk +
# target-aware PLIF plausibility pass. Muegge/Lipinski remain disclosed
# columns for Methods/SI reporting rather than hard gates. Persist the
# boolean under both the canonical manuscript name (`is_headline_hit`) and
# the older notebook-compatibility alias (`headline_hit`) so downstream
# checks never have to recreate this definition from memory.
headline_required_cols = ['consensus_pass', 'has_pains_or_brenk_alert', 'passes_plif_check']
headline_missing_cols = [c for c in headline_required_cols if c not in docking_results_final_df.columns]
if headline_missing_cols:
    raise ValueError(f'Cannot create final-prioritization flag; missing columns: {headline_missing_cols}')

headline_mask = (
    docking_results_final_df['consensus_pass'].fillna(False).astype(bool)
    & (docking_results_final_df['has_pains_or_brenk_alert'] == False)
    & (docking_results_final_df['passes_plif_check'] == True)
)
docking_results_final_df['is_headline_hit'] = headline_mask
docking_results_final_df['headline_hit'] = headline_mask
docking_results_final_df['headline_hit_definition'] = (
    'consensus_pass AND no PAINS/Brenk alert AND target-aware PLIF pass'
)

docking_results_final_path = dockres_dir / shard_name('docking_results_final.csv')
docking_results_final_df.to_csv(docking_results_final_path, index=False)
print(f'Final consolidated docking results: {len(docking_results_final_df)} rows -> {docking_results_final_path}')

n_headline = int(docking_results_final_df['is_headline_hit'].sum())
print(f'Headline hits (consensus pass + no structural alert + passes target-aware PLIF check): {n_headline}')

if len(docking_results_final_df) and 'receptor_prep_provisional' in docking_results_final_df.columns:
    provisional_headline = docking_results_final_df[
        docking_results_final_df['is_headline_hit']
        & (docking_results_final_df['receptor_prep_provisional'] == True)
    ]
    if len(provisional_headline):
        print(f'  ⚠️  {len(provisional_headline)} headline hit row(s) use receptor units with unconfirmed fusion excision boundaries; disclose as provisional until coordinate audit is resolved.')

n_headline_and_druglike = int((docking_results_final_df['is_headline_hit']
                                & (docking_results_final_df['lipinski_pass'] == True)
                                & (docking_results_final_df['muegge_pass'] == True)).sum())
print(f'Of those, {n_headline_and_druglike} also pass BOTH Lipinski and Muegge '
      f'(supplementary disclosure, not part of the headline-hit definition above).')





Final consolidated docking results: 88 rows -> /content/drive/My Drive/gpcr_benchmark/docking/docking_results/docking_results_final.csv
Headline hits (consensus pass + no structural alert + passes target-aware PLIF check): 3
  ⚠️  1 headline hit row(s) use receptor units with unconfirmed fusion excision boundaries; disclose as provisional until coordinate audit is resolved.
Of those, 2 also pass BOTH Lipinski and Muegge (supplementary disclosure, not part of the headline-hit definition above).


## Module L: Docking Validation -- Discriminative Power, Bias Checks, ML Agreement, Rediscovery

Redocking validation (Module F) proves the docking *protocol* can reproduce a
known experimental pose. It does **not** prove the docking *score* separates
true binders from non-binders, nor that the production ligand-prep route
(SMILES -> protonation -> 3D -> PDBQT) can recover known poses. Module L is
therefore the validation layer for a publication-grade docking section.

This module adds five checks:

1. **Discriminative/enrichment validation** -- dock experimentally labelled
   ChEMBL actives and inactives from notebook 3's held-out full-pool combined
   test predictions. The cap is adaptive (`N_ENRICHMENT_PER_CLASS = 50` where
   available) and the actual per-target counts are saved.
2. **Physicochemical bias audit** -- compare active vs inactive validation
   molecules for MW, cLogP, TPSA, HBD/HBA, rotatable bonds, and formal charge.
   This is important because docking benchmarks can look strong when labels
   are separable by gross properties rather than binding interactions.
3. **ROC-AUC + EF + BEDROC with bootstrap CIs** -- ROC-AUC alone is not enough
   for virtual screening because early recognition matters. EF and BEDROC are
   reported with uncertainty intervals.
4. **ML-score-vs-docking-score correlation** -- Spearman correlation on the
   enrichment set, framed as complementarity rather than a requirement for
   strong agreement.
5. **Production-pipeline rediscovery** -- for reference ligands present in the
   DrugBank library, rebuild from DrugBank SMILES through the same production
   preparation code used for candidates, dock with both Vina and GNINA, and
   report RMSD plus rank percentile against the production candidate set.

This still remains retrospective computational validation, not experimental
confirmation. The goal is to make the docking layer transparent, difficult to
game, and appropriately caveated for a JCIM-level methods/results section.


In [20]:
# =============================================================================
# ENRICHMENT VALIDATION SET -- real experimentally labelled ChEMBL actives +
# inactives from notebook 3's held-out full-pool/combined test predictions.
# =============================================================================
N_ENRICHMENT_PER_CLASS = 50
MIN_ENRICHMENT_PER_CLASS = 10
ENRICHMENT_SEED = BASE_SEED

predictions_path = results_path / 'deployed_model_test_predictions.csv'
assert predictions_path.exists(), (
    f'notebook 3 output not found at {predictions_path} -- run 03_ml_benchmark.ipynb first.')
predictions_df = pd.read_csv(predictions_path)

best_algo_path = results_path / 'best_algorithm_by_combination.csv'
assert best_algo_path.exists(), (
    f'notebook 3 output not found at {best_algo_path} -- run 03_ml_benchmark.ipynb first.')
best_algo_df = pd.read_csv(best_algo_path)

def _choose_deployed_algorithm(target):
    '''Return notebook 3's deployed full-pool/combined classifier.

    The deployed algorithm is selected upstream from leak-free inner-CV
    performance on cal_train and persisted in best_algorithm_by_combination.csv.
    Do NOT reselect here from final_model_test_performance.csv: that would pick
    the model using the held-out test set, i.e. a selection-on-test leak.
    File existence is also not a signal, because notebook 3 persists artifacts
    for all three algorithms for benchmark transparency.'''
    sub_algo = best_algo_df[
        (best_algo_df.target == target)
        & (best_algo_df.activity_pool == 'full')
        & (best_algo_df.task == 'classification')
    ]
    if len(sub_algo) == 0:
        return None
    return str(sub_algo.iloc[0]['best_algorithm'])

enrichment_records = []
enrichment_sampling_records = []
for target in design_df['target'].unique():
    algo = _choose_deployed_algorithm(target)
    if algo is None:
        print(f'  ⚠️  {target}: no deployed algorithm/test predictions found, skipping enrichment set')
        continue
    sub = predictions_df[
        (predictions_df.target == target) & (predictions_df.activity_pool == 'full')
        & (predictions_df.feature_representation == 'combined') & (predictions_df.algorithm == algo)
    ].copy()
    actives = sub[sub.y_true_class == 1]
    inactives = sub[sub.y_true_class == 0]
    n_act = min(N_ENRICHMENT_PER_CLASS, len(actives))
    n_inact = min(N_ENRICHMENT_PER_CLASS, len(inactives))
    enrichment_sampling_records.append({
        'target': target, 'algorithm_used': algo, 'available_actives': len(actives),
        'available_inactives': len(inactives), 'sampled_actives': n_act, 'sampled_inactives': n_inact,
        'meets_minimum_recommended': bool(n_act >= MIN_ENRICHMENT_PER_CLASS and n_inact >= MIN_ENRICHMENT_PER_CLASS),
    })
    if n_act == 0 or n_inact == 0:
        print(f'  ⚠️  {target}: insufficient test-set actives({len(actives)})/inactives({len(inactives)}), skipping')
        continue
    sampled_act = actives.sample(n=n_act, random_state=ENRICHMENT_SEED)
    sampled_inact = inactives.sample(n=n_inact, random_state=ENRICHMENT_SEED)
    sampled = pd.concat([sampled_act, sampled_inact], ignore_index=True)
    sampled['algorithm_used'] = algo
    enrichment_records.append(sampled)
    print(f'{target}: {n_act} actives + {n_inact} inactives sampled (deployed algorithm: {algo})')

enrichment_df = pd.concat(enrichment_records, ignore_index=True) if enrichment_records else pd.DataFrame()
enrichment_sampling_df = pd.DataFrame(enrichment_sampling_records)

# Join clean_smiles from notebook 02 cleaned data.
smiles_lookup_frames = []
for target in (enrichment_df['target'].unique() if len(enrichment_df) else []):
    cleaned_path = data_path / 'processed' / f'cleaned_data_{target}_full.csv'
    if not cleaned_path.exists():
        continue
    cdf = pd.read_csv(cleaned_path)[['clean_smiles', 'molecule_chembl_id']].copy()
    cdf['target'] = target
    smiles_lookup_frames.append(cdf)
smiles_lookup_df = pd.concat(smiles_lookup_frames, ignore_index=True) if smiles_lookup_frames else pd.DataFrame()

if len(enrichment_df) and len(smiles_lookup_df):
    enrichment_df = enrichment_df.merge(smiles_lookup_df, on=['target', 'molecule_chembl_id'], how='left')
    n_missing_smiles = enrichment_df['clean_smiles'].isna().sum()
    if n_missing_smiles:
        print(f'  ⚠️  {n_missing_smiles} enrichment compound(s) could not be matched to a SMILES, dropping')
        enrichment_df = enrichment_df.dropna(subset=['clean_smiles'])

def _mol_props(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {k: np.nan for k in ['mw', 'clogp', 'tpsa', 'hbd', 'hba', 'rot_bonds', 'formal_charge']}
    return {
        'mw': float(Descriptors.MolWt(mol)),
        'clogp': float(Crippen.MolLogP(mol)),
        'tpsa': float(Descriptors.TPSA(mol)),
        'hbd': int(Lipinski.NumHDonors(mol)),
        'hba': int(Lipinski.NumHAcceptors(mol)),
        'rot_bonds': int(Lipinski.NumRotatableBonds(mol)),
        'formal_charge': int(Chem.GetFormalCharge(mol)),
    }

if len(enrichment_df):
    props_df = pd.DataFrame([_mol_props(s) for s in enrichment_df['clean_smiles']])
    enrichment_df = pd.concat([enrichment_df.reset_index(drop=True), props_df], axis=1)

enrichment_set_path = dockres_dir / shard_name('enrichment_validation_set.csv')
enrichment_sampling_path = dockres_dir / shard_name('enrichment_sampling_summary.csv')
enrichment_df.to_csv(enrichment_set_path, index=False)
enrichment_sampling_df.to_csv(enrichment_sampling_path, index=False)
print(f'\nEnrichment validation set: {len(enrichment_df)} compounds -> {enrichment_set_path}')
if len(enrichment_df):
    print(enrichment_df.groupby(['target', 'y_true_class']).size())

# Physicochemical property-bias check between actives and inactives.
property_cols = ['mw', 'clogp', 'tpsa', 'hbd', 'hba', 'rot_bonds', 'formal_charge']
property_bias_records = []
for target, grp in enrichment_df.groupby('target') if len(enrichment_df) else []:
    for prop in property_cols:
        a = grp.loc[grp.y_true_class == 1, prop].dropna()
        i = grp.loc[grp.y_true_class == 0, prop].dropna()
        if len(a) == 0 or len(i) == 0:
            continue
        pooled = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(i)-1)*i.var(ddof=1)) / max(len(a)+len(i)-2, 1))
        std_diff = float((a.mean() - i.mean()) / pooled) if pooled and not np.isnan(pooled) else np.nan
        property_bias_records.append({
            'target': target, 'property': prop, 'active_mean': float(a.mean()), 'inactive_mean': float(i.mean()),
            'active_median': float(a.median()), 'inactive_median': float(i.median()),
            'standardized_mean_difference': std_diff,
            'abs_standardized_mean_difference': abs(std_diff) if not np.isnan(std_diff) else np.nan,
            'n_active': len(a), 'n_inactive': len(i),
        })
property_bias_df = pd.DataFrame(property_bias_records)
property_bias_path = dockres_dir / shard_name('enrichment_property_bias.csv')
property_bias_df.to_csv(property_bias_path, index=False)
print(f'Property-bias audit saved: {property_bias_path}')
if len(property_bias_df):
    display(property_bias_df.sort_values('abs_standardized_mean_difference', ascending=False).head(15))

# ---- Prepare enrichment-set ligand PDBQTs (same production function every DrugBank candidate uses) ----
enrichment_ligand_status = []
for _, row in enrichment_df.iterrows():
    cid = row['global_compound_id']
    out_pdbqt = ligand_dir / f'enrichment_{cid}.pdbqt'
    cached, cache_status = cached_ligand_or_none(out_pdbqt)
    if cached is not None:
        enrichment_ligand_status.append({'global_compound_id': cid, 'status': cache_status})
        continue
    if cache_status.startswith('stale_invalid_atom_types'):
        print(f'  ♻️  {out_pdbqt.name}: {cache_status}; regenerating')
    mol = Chem.MolFromSmiles(row['clean_smiles'])
    if mol is None:
        enrichment_ligand_status.append({'global_compound_id': cid, 'status': 'failed_parse'})
        continue
    result = prepare_ligand_pdbqt(mol, out_pdbqt)
    enrichment_ligand_status.append({'global_compound_id': cid, 'status': 'prepared' if result else ('failed_prep_after_invalid_cache' if cache_status.startswith('stale_invalid_atom_types') else 'failed_prep')})

enrichment_ligand_df = pd.DataFrame(enrichment_ligand_status)
n_ok = enrichment_ligand_df['status'].isin(['prepared', 'cached']).sum() if len(enrichment_ligand_df) else 0
print(f'Enrichment ligands ready: {n_ok} / {len(enrichment_ligand_df)}')
enrichment_ligand_path = dockres_dir / shard_name('enrichment_ligand_prep_status.csv')
enrichment_ligand_df.to_csv(enrichment_ligand_path, index=False)


drd2: 50 actives + 50 inactives sampled (deployed algorithm: XGBoost)
cb2: 50 actives + 50 inactives sampled (deployed algorithm: LightGBM)
adora2a: 50 actives + 50 inactives sampled (deployed algorithm: Random Forest)
oprm1: 50 actives + 50 inactives sampled (deployed algorithm: Random Forest)
ccr5: 50 actives + 50 inactives sampled (deployed algorithm: Random Forest)

Enrichment validation set: 500 compounds -> /content/drive/My Drive/gpcr_benchmark/docking/docking_results/enrichment_validation_set.csv
target   y_true_class
adora2a  0               50
         1               50
cb2      0               50
         1               50
ccr5     0               50
         1               50
drd2     0               50
         1               50
oprm1    0               50
         1               50
dtype: int64
Property-bias audit saved: /content/drive/My Drive/gpcr_benchmark/docking/docking_results/enrichment_property_bias.csv


,target,property,active_mean,inactive_mean,active_median,inactive_median,standardized_mean_difference,abs_standardized_mean_difference,n_active,n_inactive
8,cb2,clogp,4.066553,5.144931,4.05157,4.83100,-0.606668,0.606668,50,50
22,drd2,clogp,4.137462,3.474042,4.01246,3.47641,0.594876,0.594876,50,50
29,oprm1,clogp,4.341437,3.587305,4.36692,3.70445,0.516110,0.516110,50,50
21,drd2,mw,414.566280,372.741560,429.78900,368.98050,0.514404,0.514404,50,50
11,cb2,hba,4.400000,3.740000,4.00000,4.00000,0.469550,0.469550,50,50
15,ccr5,clogp,5.516894,4.244131,5.43426,4.53720,0.464213,0.464213,50,50
4,adora2a,hba,6.860000,5.940000,6.50000,6.00000,0.456039,0.456039,50,50
9,cb2,tpsa,73.528800,63.510000,72.89500,61.09500,0.405182,0.405182,50,50
28,oprm1,mw,452.282480,404.957469,444.06100,400.38700,0.392885,0.392885,50,50
2,adora2a,tpsa,113.760600,102.093600,109.79000,98.12000,0.358162,0.358162,50,50


Enrichment ligands ready: 497 / 500


In [21]:
# =============================================================================
# DOCK THE ENRICHMENT SET -- same Vina/GNINA calls, boxes, and validated units.
# =============================================================================
_current_enrichment_pairs = set()
if len(enrichment_df):
    for _target, _grp in enrichment_df.groupby('target'):
        _units = [f'{r[0]}_{r[1]}' for r in RUN_UNITS
                  if r[0] == _target and f'{r[0]}_{r[1]}' in units_passing_redock]
        for _unit in _units:
            for _cid in _grp['global_compound_id']:
                _current_enrichment_pairs.add((_unit, _cid))

_enrichment_vina_cols = ['unit', 'target', 'state', 'global_compound_id', 'y_true_class', 'vina_affinity', 'status']
_enrichment_gnina_cols = ['unit', 'target', 'state', 'global_compound_id', 'y_true_class', 'gnina_cnn_score', 'status']

# Live-session visibility: with up to 100 compounds x 2 engines per unit,
# the per-unit-only print/checkpoint below can go silent for a long stretch
# on Colab (unlike the cluster's log-file convention, this cell's output is
# watched directly). Print + checkpoint every N compounds too, not just once
# the whole unit finishes.
ENRICHMENT_PROGRESS_EVERY_N = 10

enrichment_vina_results = []
enrichment_vina_ckpt = dockres_dir / shard_name('enrichment_vina_results.csv')
if enrichment_vina_ckpt.exists():
    _loaded_enrich_vina = pd.read_csv(enrichment_vina_ckpt)
    _loaded_enrich_vina = _loaded_enrich_vina[
        _loaded_enrich_vina.apply(lambda r: (r['unit'], r['global_compound_id']) in _current_enrichment_pairs, axis=1)
    ]
    enrichment_vina_results = _loaded_enrich_vina.to_dict('records')
# Only a real 'ok' result counts as done -- a 'failed: ...' row must be
# retried on rerun. Applies to both engines for consistency, though this
# bug only actually bit GNINA live (Vina had zero failures in the run that
# found it).
enrichment_vina_results = [r for r in enrichment_vina_results if r.get('status') == 'ok']
_done_enrich_vina = {(r['unit'], r['global_compound_id']) for r in enrichment_vina_results}

enrichment_gnina_results = []
enrichment_gnina_ckpt = dockres_dir / shard_name('enrichment_gnina_results.csv')
if enrichment_gnina_ckpt.exists():
    _loaded_enrich_gnina = pd.read_csv(enrichment_gnina_ckpt)
    _loaded_enrich_gnina = _loaded_enrich_gnina[
        _loaded_enrich_gnina.apply(lambda r: (r['unit'], r['global_compound_id']) in _current_enrichment_pairs, axis=1)
    ]
    enrichment_gnina_results = _loaded_enrich_gnina.to_dict('records')
# Bug found live: without this filter, the 903-compound GNINA outage from
# the enrichment run would become permanent on any rerun -- every failed
# pair already has a checkpoint row and would never be re-attempted.
enrichment_gnina_results = [r for r in enrichment_gnina_results if r.get('status') == 'ok']
_done_enrich_gnina = {(r['unit'], r['global_compound_id']) for r in enrichment_gnina_results}

for target, state, pdb_id, ref_ligand_code, pocket_type, species, notes in RUN_UNITS:
    unit_key = f'{target}_{state}'
    if unit_key not in units_passing_redock:
        continue
    receptor_pdbqt = protein_dir / f'{unit_key}_{pdb_id}_prepared.pdbqt'
    box = box_definitions[unit_key]
    target_enrichment = enrichment_df[enrichment_df.target == target] if len(enrichment_df) else enrichment_df

    for _compound_i, (_, cand) in enumerate(target_enrichment.iterrows(), start=1):
        cid = cand['global_compound_id']
        ligand_pdbqt = ligand_dir / f'enrichment_{cid}.pdbqt'
        if not ligand_pdbqt.exists():
            continue
        bad_types = invalid_pdbqt_atom_types(ligand_pdbqt)
        if bad_types:
            print(f'  ⚠️  {ligand_pdbqt.name}: invalid PDBQT atom types {bad_types[:5]}, skipping docking until ligand prep is rerun')
            continue

        if (unit_key, cid) not in _done_enrich_vina:
            out_path = dockres_dir / f'enrich_vina_{unit_key}_{cid}.pdbqt'
            try:
                run_vina_redock(receptor_pdbqt, ligand_pdbqt, box, BASE_SEED, VINA_EXHAUSTIVENESS, out_path)
                best_affinity = None
                with open(out_path) as f:
                    for line in f:
                        if line.startswith('REMARK VINA RESULT'):
                            best_affinity = float(line.split()[3]); break
                enrichment_vina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                                 'global_compound_id': cid, 'y_true_class': int(cand['y_true_class']),
                                                 'vina_affinity': best_affinity, 'status': 'ok'})
            except Exception as exc:
                enrichment_vina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                                 'global_compound_id': cid, 'y_true_class': int(cand['y_true_class']),
                                                 'vina_affinity': None, 'status': f'failed: {exc}'})

        if (unit_key, cid) not in _done_enrich_gnina:
            out_path = dockres_dir / f'enrich_gnina_{unit_key}_{cid}.sdf'
            try:
                run_gnina(receptor_pdbqt, ligand_pdbqt, box, out_path)
                cnn_score = None
                for mol in Chem.SDMolSupplier(str(out_path), sanitize=False):
                    if mol is not None and mol.HasProp('CNNscore'):
                        cnn_score = float(mol.GetProp('CNNscore')); break
                enrichment_gnina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                                  'global_compound_id': cid, 'y_true_class': int(cand['y_true_class']),
                                                  'gnina_cnn_score': cnn_score, 'status': 'ok'})
            except Exception as exc:
                enrichment_gnina_results.append({'unit': unit_key, 'target': target, 'state': state,
                                                  'global_compound_id': cid, 'y_true_class': int(cand['y_true_class']),
                                                  'gnina_cnn_score': None, 'status': f'failed: {exc}'})

        if _compound_i % ENRICHMENT_PROGRESS_EVERY_N == 0 or _compound_i == len(target_enrichment):
            pd.DataFrame(enrichment_vina_results, columns=_enrichment_vina_cols).drop_duplicates(
                subset=['unit', 'global_compound_id'], keep='last').to_csv(enrichment_vina_ckpt, index=False)
            pd.DataFrame(enrichment_gnina_results, columns=_enrichment_gnina_cols).drop_duplicates(
                subset=['unit', 'global_compound_id'], keep='last').to_csv(enrichment_gnina_ckpt, index=False)
            print(f'  {unit_key}: {_compound_i} / {len(target_enrichment)} enrichment compounds docked')

    pd.DataFrame(enrichment_vina_results, columns=_enrichment_vina_cols).drop_duplicates(
        subset=['unit', 'global_compound_id'], keep='last').to_csv(enrichment_vina_ckpt, index=False)
    pd.DataFrame(enrichment_gnina_results, columns=_enrichment_gnina_cols).drop_duplicates(
        subset=['unit', 'global_compound_id'], keep='last').to_csv(enrichment_gnina_ckpt, index=False)
    print(f'{unit_key}: enrichment docking complete for {len(target_enrichment)} compounds')

enrichment_vina_df = pd.DataFrame(enrichment_vina_results, columns=_enrichment_vina_cols).drop_duplicates(subset=['unit', 'global_compound_id'], keep='last')
enrichment_gnina_df = pd.DataFrame(enrichment_gnina_results, columns=_enrichment_gnina_cols).drop_duplicates(subset=['unit', 'global_compound_id'], keep='last')
print(f'\nEnrichment Vina results: {len(enrichment_vina_df)} | GNINA results: {len(enrichment_gnina_df)}')



  drd2_active: 10 / 100 enrichment compounds docked
  drd2_active: 20 / 100 enrichment compounds docked
  drd2_active: 30 / 100 enrichment compounds docked
  drd2_active: 40 / 100 enrichment compounds docked
  drd2_active: 50 / 100 enrichment compounds docked
  drd2_active: 60 / 100 enrichment compounds docked
  drd2_active: 70 / 100 enrichment compounds docked
  drd2_active: 80 / 100 enrichment compounds docked
  drd2_active: 90 / 100 enrichment compounds docked
  drd2_active: 100 / 100 enrichment compounds docked
drd2_active: enrichment docking complete for 100 compounds
  drd2_inactive: 10 / 100 enrichment compounds docked
  drd2_inactive: 20 / 100 enrichment compounds docked
  drd2_inactive: 30 / 100 enrichment compounds docked
  drd2_inactive: 40 / 100 enrichment compounds docked
  drd2_inactive: 50 / 100 enrichment compounds docked
  drd2_inactive: 60 / 100 enrichment compounds docked
  drd2_inactive: 70 / 100 enrichment compounds docked
  drd2_inactive: 80 / 100 enrichment compo

In [22]:
# =============================================================================
# DISCRIMINATIVE VALIDATION -- ROC-AUC, EF, BEDROC, with bootstrap CIs.
# =============================================================================
from rdkit.ML.Scoring import Scoring
from sklearn.metrics import roc_auc_score

BEDROC_ALPHA = 20.0
EF_FRACTIONS = [0.01, 0.05, 0.10]
N_BOOTSTRAP_VALIDATION = 1000
BOOTSTRAP_SEED = BASE_SEED

def _score_metrics(df, score_col, higher_is_better):
    sub = df.dropna(subset=[score_col, 'y_true_class']).copy()
    if sub['y_true_class'].nunique() < 2 or len(sub) < 4:
        return None
    ordered = sub.sort_values(score_col, ascending=not higher_is_better)
    scores = [[bool(v)] for v in ordered['y_true_class'].astype(bool)]
    auc = roc_auc_score(sub['y_true_class'], sub[score_col] if higher_is_better else -sub[score_col])
    bedroc = Scoring.CalcBEDROC(scores, 0, BEDROC_ALPHA)
    ef = Scoring.CalcEnrichment(scores, 0, EF_FRACTIONS)
    return {'auc': float(auc), 'bedroc': float(bedroc),
            'ef_1pct': float(ef[0]), 'ef_5pct': float(ef[1]), 'ef_10pct': float(ef[2]),
            'n_compounds': len(sub), 'n_actives': int(sub.y_true_class.sum()),
            'n_inactives': int((sub.y_true_class == 0).sum())}

def _bootstrap_metric_cis(df, score_col, higher_is_better, n_boot=N_BOOTSTRAP_VALIDATION, seed=BOOTSTRAP_SEED):
    sub = df.dropna(subset=[score_col, 'y_true_class']).copy()
    if sub['y_true_class'].nunique() < 2 or len(sub) < 8:
        return {}
    rng = np.random.default_rng(seed)
    vals = []
    arr_idx = np.arange(len(sub))
    for _ in range(n_boot):
        sample = sub.iloc[rng.choice(arr_idx, size=len(sub), replace=True)]
        if sample['y_true_class'].nunique() < 2:
            continue
        m = _score_metrics(sample, score_col, higher_is_better)
        if m:
            vals.append(m)
    if not vals:
        return {}
    bdf = pd.DataFrame(vals)
    ci = {}
    for metric in ['auc', 'bedroc', 'ef_1pct', 'ef_5pct', 'ef_10pct']:
        ci[f'{metric}_ci_low'] = float(bdf[metric].quantile(0.025))
        ci[f'{metric}_ci_high'] = float(bdf[metric].quantile(0.975))
    ci['n_bootstrap_successful'] = int(len(bdf))
    return ci

discriminative_records = []
for score_df, score_col, score_name, higher in [
    (enrichment_vina_df, 'vina_affinity', 'vina_affinity', False),
    (enrichment_gnina_df, 'gnina_cnn_score', 'gnina_cnn_score', True),
]:
    if not len(score_df):
        continue
    for unit_key, grp in score_df.groupby('unit'):
        m = _score_metrics(grp, score_col, higher)
        if m:
            m.update(_bootstrap_metric_cis(grp, score_col, higher))
            discriminative_records.append({'unit': unit_key, 'score_type': score_name, **m})

discriminative_df = pd.DataFrame(discriminative_records)
discriminative_path = dockres_dir / shard_name('discriminative_validation.csv')
discriminative_df.to_csv(discriminative_path, index=False)
print(f'Discriminative validation saved: {discriminative_path}')
print(discriminative_df.to_string(index=False) if len(discriminative_df) else '(no unit had enough compounds/classes)')
print(f'\nBEDROC alpha={BEDROC_ALPHA}; EF fractions={EF_FRACTIONS}; bootstrap iterations={N_BOOTSTRAP_VALIDATION}.')


Discriminative validation saved: /content/drive/My Drive/gpcr_benchmark/docking/docking_results/discriminative_validation.csv
                               unit      score_type      auc   bedroc  ef_1pct  ef_5pct  ef_10pct  n_compounds  n_actives  n_inactives  auc_ci_low  auc_ci_high  bedroc_ci_low  bedroc_ci_high  ef_1pct_ci_low  ef_1pct_ci_high  ef_5pct_ci_low  ef_5pct_ci_high  ef_10pct_ci_low  ef_10pct_ci_high  n_bootstrap_successful
                     adora2a_active   vina_affinity 0.522653 0.666456 2.020408 1.616327  1.212245           99         49           50    0.410679     0.628318       0.335180        0.882704        0.000000         2.538462        0.459953         2.302326         0.421057          1.782450                    1000
                   adora2a_inactive   vina_affinity 0.591020 0.561982 0.000000 1.212245  1.010204           99         49           50    0.475844     0.700833       0.272560        0.869543        0.000000         2.357143        0.380769   

In [23]:
# =============================================================================
# ML-SCORE VS DOCKING-SCORE CORRELATION -- Spearman, on enrichment compounds.
# =============================================================================
from scipy.stats import spearmanr

corr_records = []
enrichment_with_proba = (enrichment_df[['global_compound_id', 'target', 'predicted_proba', 'algorithm_used']].copy()
                          if len(enrichment_df) else pd.DataFrame())

for score_df, score_col, score_name in [
    (enrichment_vina_df, 'vina_affinity', 'vina_affinity'),
    (enrichment_gnina_df, 'gnina_cnn_score', 'gnina_cnn_score'),
]:
    if not len(score_df) or not len(enrichment_with_proba):
        continue
    merged = score_df.merge(enrichment_with_proba, on=['global_compound_id', 'target'], how='left')
    for unit_key, grp in merged.groupby('unit'):
        sub = grp.dropna(subset=[score_col, 'predicted_proba'])
        if len(sub) < 4:
            continue
        rho, p = spearmanr(sub[score_col], sub['predicted_proba'])
        algorithms_used = sorted(sub['algorithm_used'].dropna().astype(str).unique()) if 'algorithm_used' in sub.columns else []
        corr_records.append({'unit': unit_key, 'score_type': score_name, 'n_compounds': len(sub),
                              'algorithm_used': ';'.join(algorithms_used),
                              'algorithm_provenance': 'stamped from enrichment_validation_set.csv at correlation time',
                              'spearman_rho': float(rho), 'spearman_p': float(p)})

correlation_df = pd.DataFrame(corr_records)
correlation_path = dockres_dir / shard_name('ml_docking_correlation.csv')
correlation_df.to_csv(correlation_path, index=False)
print(f'ML-vs-docking Spearman correlation saved: {correlation_path}')
print(correlation_df.to_string(index=False) if len(correlation_df) else '(no unit had enough paired compounds)')


ML-vs-docking Spearman correlation saved: /content/drive/My Drive/gpcr_benchmark/docking/docking_results/ml_docking_correlation.csv
                               unit      score_type  n_compounds algorithm_used                                           algorithm_provenance  spearman_rho   spearman_p
                     adora2a_active   vina_affinity           99  Random Forest stamped from enrichment_validation_set.csv at correlation time     -0.045164 6.571179e-01
                   adora2a_inactive   vina_affinity           99  Random Forest stamped from enrichment_validation_set.csv at correlation time     -0.090545 3.727682e-01
                         cb2_active   vina_affinity          100       LightGBM stamped from enrichment_validation_set.csv at correlation time     -0.343966 4.583711e-04
                       cb2_inactive   vina_affinity          100       LightGBM stamped from enrichment_validation_set.csv at correlation time     -0.087309 3.877161e-01
ccr5_inactive_allo

In [24]:
# =============================================================================
# DOCKING-SIDE REDISCOVERY CHECK -- production ligand-prep route, both engines.
# =============================================================================
REDISCOVERY_CHECK_UNITS = {
    'drd2_active': 'DB01200',     # bromocriptine (bound as 08Y in 6VMS)
    'adora2a_active': 'DB03719',  # NECA (bound as NEC in 5G53)
    'oprm1_active': 'DB14030',    # PZM21 (bound as 8QY in 8EFO)
}

def _gnina_pose_to_pdbqt(sdf_path: Path, pdbqt_path: Path):
    subprocess.run(['obabel', str(sdf_path), '-O', str(pdbqt_path)], check=True, capture_output=True)
    return pdbqt_path

def _rank_percentile_against_candidates(unit_key, drugbank_id, vina_score=None, gnina_score=None):
    out = {}
    if vina_score is not None and len(vina_df):
        sub = vina_df[vina_df.unit == unit_key].dropna(subset=['vina_affinity'])
        if len(sub):
            out['vina_rank_percentile_vs_candidates'] = float((sub['vina_affinity'] <= vina_score).mean())
    if gnina_score is not None and len(gnina_df):
        sub = gnina_df[gnina_df.unit == unit_key].dropna(subset=['gnina_cnn_score'])
        if len(sub):
            out['gnina_rank_percentile_vs_candidates'] = float((sub['gnina_cnn_score'] >= gnina_score).mean())
    return out

drugbank_library_path = screen_path / 'drugbank_library_standardised.csv'
rediscovery_records = []
if not drugbank_library_path.exists():
    print(f'  ⚠️  {drugbank_library_path} not found -- skipping docking-side rediscovery check entirely')
else:
    drugbank_library_df = pd.read_csv(drugbank_library_path)
    for unit_key, drugbank_id in REDISCOVERY_CHECK_UNITS.items():
        if unit_key not in units_passing_redock:
            print(f'  ⏭️  {unit_key}: did not pass redocking validation, skipping rediscovery check')
            continue
        match = drugbank_library_df[drugbank_library_df.drugbank_id == drugbank_id]
        if len(match) == 0:
            rediscovery_records.append({'unit': unit_key, 'drugbank_id': drugbank_id, 'status': 'not_found_in_library'})
            print(f'  ❌ {unit_key}: {drugbank_id} not found in {drugbank_library_path.name}')
            continue

        target, state, pdb_id, ref_ligand_code = next(
            (r[0], r[1], r[2], r[3]) for r in RUN_UNITS if f'{r[0]}_{r[1]}' == unit_key)
        mol = Chem.MolFromSmiles(match.iloc[0]['clean_smiles'])
        if mol is None:
            rediscovery_records.append({'unit': unit_key, 'drugbank_id': drugbank_id, 'status': 'failed_smiles_parse'})
            continue
        rediscovery_pdbqt = ligand_dir / f'rediscovery_{unit_key}_{drugbank_id}.pdbqt'
        prep_result, cache_status = cached_ligand_or_none(rediscovery_pdbqt)
        if prep_result is None:
            if cache_status.startswith('stale_invalid_atom_types'):
                print(f'  ♻️  {rediscovery_pdbqt.name}: {cache_status}; regenerating')
            prep_result = prepare_ligand_pdbqt(mol, rediscovery_pdbqt)
        if prep_result is None:
            rediscovery_records.append({'unit': unit_key, 'drugbank_id': drugbank_id, 'status': 'failed_ligand_prep'})
            continue

        receptor_pdbqt = protein_dir / f'{unit_key}_{pdb_id}_prepared.pdbqt'
        reference_pdbqt = ligand_dir / f'{unit_key}_{pdb_id}_{ref_ligand_code}_reference.pdbqt'
        box = box_definitions.get(unit_key)
        if box is None or not receptor_pdbqt.exists() or not reference_pdbqt.exists():
            rediscovery_records.append({'unit': unit_key, 'drugbank_id': drugbank_id, 'status': 'missing_receptor_or_reference'})
            continue

        record = {'unit': unit_key, 'drugbank_id': drugbank_id, 'status': 'ok'}
        try:
            vina_pose = dockres_dir / f'rediscovery_vina_{unit_key}_{drugbank_id}.pdbqt'
            run_vina_redock(receptor_pdbqt, rediscovery_pdbqt, box, BASE_SEED, VINA_EXHAUSTIVENESS, vina_pose)
            vina_score = None
            with open(vina_pose) as f:
                for line in f:
                    if line.startswith('REMARK VINA RESULT'):
                        vina_score = float(line.split()[3]); break
            record.update({
                'vina_affinity': vina_score,
                'vina_rmsd_vs_crystal_reference': compute_heavy_atom_rmsd(vina_pose, reference_pdbqt),
                'vina_n_protein_contacts': count_protein_contacts(vina_pose, receptor_pdbqt),
            })
        except Exception as exc:
            record['vina_error'] = str(exc)

        try:
            gnina_sdf = dockres_dir / f'rediscovery_gnina_{unit_key}_{drugbank_id}.sdf'
            run_gnina(receptor_pdbqt, rediscovery_pdbqt, box, gnina_sdf)
            gnina_score = None
            for gm in Chem.SDMolSupplier(str(gnina_sdf), sanitize=False):
                if gm is not None and gm.HasProp('CNNscore'):
                    gnina_score = float(gm.GetProp('CNNscore')); break
            gnina_pdbqt = _gnina_pose_to_pdbqt(gnina_sdf, dockres_dir / f'rediscovery_gnina_{unit_key}_{drugbank_id}.pdbqt')
            record.update({
                'gnina_cnn_score': gnina_score,
                'gnina_rmsd_vs_crystal_reference': compute_heavy_atom_rmsd(gnina_pdbqt, reference_pdbqt),
                'gnina_n_protein_contacts': count_protein_contacts(gnina_pdbqt, receptor_pdbqt),
            })
        except Exception as exc:
            record['gnina_error'] = str(exc)

        record.update(_rank_percentile_against_candidates(unit_key, drugbank_id,
                                                          record.get('vina_affinity'), record.get('gnina_cnn_score')))
        record['vina_passed_rmsd_threshold'] = bool(record.get('vina_rmsd_vs_crystal_reference', np.inf) <= RMSD_PASS_THRESHOLD)
        record['gnina_passed_rmsd_threshold'] = bool(record.get('gnina_rmsd_vs_crystal_reference', np.inf) <= RMSD_PASS_THRESHOLD)
        record['overall_rediscovery_pass'] = bool(record['vina_passed_rmsd_threshold'] and record['gnina_passed_rmsd_threshold'])
        rediscovery_records.append(record)
        print(f"  {unit_key}/{drugbank_id}: Vina RMSD={record.get('vina_rmsd_vs_crystal_reference')}, GNINA RMSD={record.get('gnina_rmsd_vs_crystal_reference')}")

rediscovery_df = pd.DataFrame(rediscovery_records)
rediscovery_path = dockres_dir / shard_name('docking_side_rediscovery_check.csv')
rediscovery_df.to_csv(rediscovery_path, index=False)
print(f'\nDocking-side rediscovery check saved: {rediscovery_path}')
print(rediscovery_df.to_string(index=False) if len(rediscovery_df) else '(no units checked)')


  drd2_active/DB01200: Vina RMSD=1.1214679368257041, GNINA RMSD=0.586310755582427
  adora2a_active/DB03719: Vina RMSD=1.1683597130177856, GNINA RMSD=1.9442971578620154
  oprm1_active/DB14030: Vina RMSD=1.3914049167428544, GNINA RMSD=2.540265985697261

Docking-side rediscovery check saved: /content/drive/My Drive/gpcr_benchmark/docking/docking_results/docking_side_rediscovery_check.csv
          unit drugbank_id status  vina_affinity  vina_rmsd_vs_crystal_reference  vina_n_protein_contacts  gnina_cnn_score  gnina_rmsd_vs_crystal_reference  gnina_n_protein_contacts  vina_rank_percentile_vs_candidates  gnina_rank_percentile_vs_candidates  vina_passed_rmsd_threshold  gnina_passed_rmsd_threshold  overall_rediscovery_pass
   drd2_active     DB01200     ok        -12.974                        1.121468                      151         0.939009                         0.586311                       165                                 0.0                                  0.0                    

In [25]:
# =============================================================================
# MANIFEST -- SHA-256-hashed, same convention as notebooks 2-4.
# =============================================================================
def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def _git_commit():
    try:
        return subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(PROJECT_DIR),
                                        stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None

manifest_outputs = {}
for name, path in [
    ('design_matrix.csv', None), ('receptor_prep_status.csv', receptor_prep_path),
    ('docking_candidates.csv', docking_candidates_path),
    ('docking_candidate_selection_summary.csv', docking_candidate_selection_summary_path),
    ('redocking_validation.csv', dockres_dir / shard_name('redocking_validation.csv')),
    ('redocking_all_attempts.csv', dockres_dir / shard_name('redocking_all_attempts.csv')),
    ('grid_box_definitions.csv', box_path),
    ('vina_docking_results.csv', vina_ckpt_path),
    ('gnina_docking_results.csv', gnina_ckpt_path),
    ('docking_consensus.csv', dockres_dir / shard_name('docking_consensus.csv')),
    ('plif_plausibility_check.csv', dockres_dir / shard_name('plif_plausibility_check.csv')),
    ('target_aware_plif_plausibility_check.csv', dockres_dir / shard_name('target_aware_plif_plausibility_check.csv')),
    ('docking_results_final.csv', docking_results_final_path),
    ('enrichment_validation_set.csv', dockres_dir / shard_name('enrichment_validation_set.csv')),
    ('enrichment_sampling_summary.csv', dockres_dir / shard_name('enrichment_sampling_summary.csv')),
    ('enrichment_property_bias.csv', dockres_dir / shard_name('enrichment_property_bias.csv')),
    ('enrichment_ligand_prep_status.csv', dockres_dir / shard_name('enrichment_ligand_prep_status.csv')),
    ('enrichment_vina_results.csv', dockres_dir / shard_name('enrichment_vina_results.csv')),
    ('enrichment_gnina_results.csv', dockres_dir / shard_name('enrichment_gnina_results.csv')),
    ('discriminative_validation.csv', dockres_dir / shard_name('discriminative_validation.csv')),
    ('ml_docking_correlation.csv', dockres_dir / shard_name('ml_docking_correlation.csv')),
    ('docking_side_rediscovery_check.csv', dockres_dir / shard_name('docking_side_rediscovery_check.csv')),
]:
    if path is not None and Path(path).exists():
        manifest_outputs[name] = {'path': str(path), 'sha256': _sha256(path), 'n_rows': sum(1 for _ in open(path)) - 1}

manifest = {
    'timestamp': datetime.datetime.now().isoformat(),
    'git_commit': _git_commit(),
    'python_version': platform.python_version(),
    'config': {
        'design_matrix_units': len(DESIGN_MATRIX), 'run_units': [f'{r[0]}_{r[1]}' for r in RUN_UNITS],
        'max_redock_attempts': MAX_REDOCK_ATTEMPTS, 'rmsd_pass_threshold': RMSD_PASS_THRESHOLD,
        'grid_box_size': GRID_BOX_SIZE, 'consensus_top_percent': CONSENSUS_TOP_PERCENT,
        'n_candidates_per_target': N_CANDIDATES_PER_TARGET,
        'candidate_source_description': CANDIDATE_SOURCE_DESCRIPTION,
        'primary_screen_activity_pool': 'full',
        'primary_screen_feature_representation': 'combined',
        'primary_screen_model_policy': 'target-specific deployed classifier from notebook 3',
        'cross_representation_consensus_role': 'full-pool sensitivity annotation only; not an inclusion gate',
        'na_policy_targets': list(NA_POLICY_TARGETS), 'na_retention_rule': NA_RETENTION_RULE,
        'n_enrichment_per_class': N_ENRICHMENT_PER_CLASS,
        'minimum_recommended_enrichment_per_class': MIN_ENRICHMENT_PER_CLASS,
        'bedroc_alpha': BEDROC_ALPHA, 'ef_fractions': EF_FRACTIONS,
        'n_bootstrap_validation': N_BOOTSTRAP_VALIDATION,
        'rediscovery_check_units': REDISCOVERY_CHECK_UNITS,
        'target_plif_rules': TARGET_PLIF_RULES,
        'generic_contact_cutoff': GENERIC_CONTACT_CUTOFF,
        'polar_contact_cutoff': POLAR_CONTACT_CUTOFF,
        'ionic_contact_cutoff': IONIC_CONTACT_CUTOFF,
        'hydrophobic_contact_cutoff': HYDROPHOBIC_CONTACT_CUTOFF,
    },
    'outputs': manifest_outputs,
    'package_versions': {},
}
# Every third-party dependency this notebook actually uses -- including
# meeko and vina, which are never `import`ed in Python code (only invoked
# as CLI subprocesses: mk_prepare_ligand.py, vina) but whose pip-installed
# versions matter just as much for reproducibility as the ones that are.
for pkg in ('rdkit', 'numpy', 'pandas', 'meeko', 'gemmi', 'dimorphite_dl',
            'pdbfixer', 'openmm', 'vina'):
    try:
        manifest['package_versions'][pkg] = getattr(__import__(pkg), '__version__', 'unknown')
    except ImportError:
        pass

# CLI-only engines have no Python-importable version at all -- gnina
# isn't even a pip package (fetched as a GitHub release binary), so no
# package-version loop could ever catch it regardless of what's in the
# tuple above. Captured separately via each binary's own --version output.
manifest['cli_versions'] = {}
for exe, version_flag in [('gnina', '--version'), ('obabel', '-V'), ('vina', '--version')]:
    exe_path = shutil.which(exe)
    if exe_path is None:
        continue
    try:
        result = subprocess.run([exe_path, version_flag], capture_output=True, text=True)
        manifest['cli_versions'][exe] = (result.stdout or result.stderr).strip().splitlines()[0]
    except Exception:
        pass

manifest_path = dockres_dir / shard_name('manifest_05_docking.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print(f'Manifest saved: {manifest_path}')
print('\nNotebook 5 (docking) complete.')




Manifest saved: /content/drive/My Drive/gpcr_benchmark/docking/docking_results/manifest_05_docking.json

Notebook 5 (docking) complete.
